# NB04 · Motor vectorial: requisitos, humo, esquema e ingesta

Hasta aquí los vectores han vivido en memoria y en ficheros `.npy`. Este notebook los pone en una base de datos, y para eso hay que elegir cuál.

### El orden: requisitos → motor → índice

Podría parecer que primero se elige el índice y después la base que lo contiene —es el orden en que se estudian—, pero al **elegir** la relación se invierte: el índice no es algo que uno monta, es algo que el motor concede. Unos dejan escoger la familia de algoritmo, otros dan grafo y dejan ajustarlo, y otros no dejan ni verlo.

Decidir "quiero tal índice" antes que el motor solo tendría sentido si ese requisito **descartara motores**, y aquí no lo hace: 15.000 vectores de 768 dimensiones son 46 MB, caben en memoria con holgura y hasta la búsqueda exacta es cuestión de milisegundos. Lo que sí los descarta son las responsabilidades operativas —filtrar, no duplicar al reingerir, sobrevivir a un reinicio, aplicar altas y bajas—, y por eso van primero.

La sección **A** escribe esos requisitos **antes** de mirar ningún motor: si se eligiera primero y luego se ajustaran al ganador, la elección no demostraría nada.

| Marca | Corpus | Fichero |
|---|---|---|
| 🔬 **MUESTRA** | 1.500 registros | `catalogo_muestra.csv` |
| 📚 **COMPLETO** | 15.000 registros | `catalogo_productos.csv` |

> 📄 Como en los notebooks anteriores, la primera línea de cada celda de código dice qué datos usa.

**Aquí el corpus por defecto es el completo**, al revés que en NB02 y NB03: lo que se decide en este notebook —qué se guarda, cómo se filtra, cuánto ocupa— es sobre el índice que se construye de verdad, que contiene los 15.000. La muestra reaparece en la prueba de humo, que es donde conviene equivocarse barato.

In [1]:
# 📄 DATOS · 📚 catalogo_productos.csv (15.000) · 🔬 catalogo_muestra.csv (1.500)
#            · consultas_filtradas.csv (4 consultas con filtro de marca)
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "src"))

import pandas as pd

from aurum.almacen import (
    NULL_POLICIES,
    PAYLOAD_SCHEMAS,
    add_normalized_key,
    batch_footprint,
    build_payload,
    combined_filter_selectivity,
    field_byte_profile,
    filter_field_profile,
    filter_reach,
    filter_writing_robustness,
    index_footprint,
    payload_budget,
    robustness_summary,
)
from aurum.datos import load_csv, normalize_brand, strip_accents, value_frequency

DATA = Path("..") / "data"
completo = load_csv(DATA / "catalogo_productos.csv")
muestra = load_csv(DATA / "catalogo_muestra.csv")
filtradas = load_csv(DATA / "consultas_filtradas.csv")

DIM = 768                     # R02: gemini-embedding-2 truncado y renormalizado
TOP_K = 10
NORMALIZACION = "unaccent"    # D03: minúsculas y sin tildes, al buscar
CAMPOS_FILTRABLES = ["brand", "color"]

print(f"📚 completo: {len(completo)} · 🔬 muestra: {len(muestra)}")
print(f"consultas con filtro: {len(filtradas)} — todas por {filtradas['filter_field'].unique()}")

📚 completo: 15000 · 🔬 muestra: 1500
consultas con filtro: 4 — todas por <StringArray>
['brand']
Length: 1, dtype: str


---

# A · Los requisitos, escritos antes de mirar ningún motor

La lista sale de tres sitios: el guion de selección de una base vectorial de la sesión 3, lo que necesitan los notebooks posteriores de este trabajo, y lo que el enunciado exige sin margen.

Buena parte no se decide: **se mide**. La celda siguiente calcula esa parte.

In [2]:
# 📄 DATOS · 📚 catalogo_productos.csv (15.000) + consultas_filtradas.csv
# Los requisitos que son un hecho del problema, no una preferencia. Se calculan
# aquí para que la tabla de abajo no lleve ni un número escrito a mano.
huella = index_footprint(len(completo), DIM)
marcas_pedidas = filtradas["filter_value"].tolist()
alcance_marca = filter_reach(completo, marcas_pedidas, field="brand")
selectividad = (
    alcance_marca.query("modo == 'raw'")
    .assign(pct_catalogo=lambda d: (100 * d["n_productos"] / len(completo)).round(2))
    [["filtro", "n_productos", "pct_catalogo"]]
)

print(f"vectores            : {huella['n_points']:,} de {huella['dim']} dimensiones")
print(f"memoria de vectores : {huella['mb_vectores']} MB")
print(f"resultados por consulta: {TOP_K}")
print()
print("Selectividad de los filtros que exige el enunciado:")
selectividad

vectores            : 15,000 de 768 dimensiones
memoria de vectores : 43.95 MB
resultados por consulta: 10

Selectividad de los filtros que exige el enunciado:


,filtro,n_productos,pct_catalogo
0,Einhell,30,0.20
1,Apple,100,0.67
2,NIKE,295,1.97
3,SAMSUNG,155,1.03


### A.1 · Lo que ya está fijado

**De recuperación:**

| Requisito | Valor | De dónde sale |
|---|---|---|
| Volumen y crecimiento | 15.000 vectores; la secuencia de eventos deja otros 15.000 (ocho altas, ocho bajas) | Catálogo y eventos |
| Representación | 768 dimensiones, decimales de 4 bytes, norma 1 | R02 y D10 |
| Un vector por producto | Sí, nunca varios | D07: ninguna ficha se acerca a la ventana del modelo |
| Resultados por consulta | 10 | Enunciado |
| Campos por los que se filtra | Marca (obligatorio) y color (añadido) | Consultas filtradas + decisión propia |
| Búsqueda dispersa o híbrida | No se le exige al motor | Si se probara el híbrido, se fusionaría por posición fuera del motor: no le pide nada |
| Cuánto cambia el corpus | 24 eventos, aplicados dos veces | Enunciado |

**De operación:**

| Requisito | Valor |
|---|---|
| Carga | Mínima: un experimento reproducible, no un servicio con tráfico |
| Memoria de la máquina | 7,9 GB, compartidos con el notebook. Los motores, de uno en uno |
| Presupuesto | Cero recurrente: quien corrija reproduce el trabajo y no puede heredar una factura |
| Despliegue | Docker, con los ficheros de la sesión 3 adaptados a este repo |

**Y lo que el enunciado impone sin margen:** SDK nativo del motor —ninguna capa de abstracción sustituyendo la configuración—, filtro ejecutado por el motor, ingesta repetible sin duplicar, persistencia real y mutaciones cuya visibilidad se pueda comprobar.

### A.2 · Lo que se aparca a propósito

El **recall mínimo aceptable** y la **latencia máxima** no se fijan aquí: se declaran justo antes de ver la curva de fidelidad, que es NB06. Anotarlos ahora invitaría a ajustarlos luego a lo que salga.

Se descartan por no aplicar a una práctica en una sola máquina: caudal de peticiones concurrentes, alta disponibilidad, tiempo de recuperación ante desastre y reparto entre varias máquinas.

---

# B · El campo extra: filtrar también por color

El enunciado solo pide filtrar por marca. Añadir el color es una mejora propia, y se somete a la misma evidencia que el resto en vez de darse por buena porque suene bien.

La pregunta no es *"¿se puede filtrar por color?"* sino **"¿es el color la clase de campo sobre la que un filtro de igualdad significa algo?"**. Un campo de vocabulario cerrado —diez colores repetidos miles de veces— se filtra con igualdad y no hay más que hablar; uno de texto libre se comporta de otra manera, y conviene saberlo **antes** de prometer un filtro que devuelve la mitad de lo que debería.

### Cómo leer la tabla siguiente

Una fila por campo filtrable. Las tres últimas columnas son el veredicto *taxonomía vs. texto libre*, y las tres apuntan en la misma dirección: **cuanto más altas, menos sirve un filtro de igualdad**.

| Columna | Qué mide | Alto significa |
|---|---|---|
| `pct_valores_unicos` | valores distintos que aparecen en un solo producto | cola larga: cada quien escribió lo suyo |
| `pct_compuestos` | productos cuyo valor lleva un separador dentro (`Negro/Rojo`) | la igualdad estricta los deja fuera |
| `pct_multipalabra` | productos con dos o más palabras (`azul marino`) | pedir una sola palabra no los encuentra |

La comparación que importa es **entre las dos filas**: `brand` y `color` deberían comportarse distinto, y si no lo hacen, la mejora del color no está justificada.

In [ ]:
# 📄 DATOS · 📚 catalogo_productos.csv (15.000)
# ¿Taxonomía o texto libre? Un `pct_valores_unicos` alto significa cola larga:
# la mayoría de los valores los escribió una sola persona una sola vez.
filter_field_profile(completo, CAMPOS_FILTRABLES)

In [ ]:
# 📄 DATOS · 📚 catalogo_productos.csv (15.000)
# Los valores más frecuentes de cada campo filtrable, para ver de qué hablamos.
display(value_frequency(completo, "color").head(8))
display(value_frequency(completo, "brand").head(5))

## B.1 · Igualdad frente a *contiene*

Si el campo es texto libre, un valor puede llevar varios dentro —`"Negro/Rojo"`, `"Negro (Black)"`— o ser de dos palabras —`"azul marino"`—: con igualdad estricta, quien pida negro no ve los dos primeros y quien pida azul no ve el tercero.

La celda mide las dos políticas sobre los mismos valores y los mismos tres modos de normalización, para no mezclar dos cambios a la vez.

### Cómo leer la tabla

Cada **fila** es un color pedido bajo un modo de normalización; cada **columna**, una política de comparación. El número son productos alcanzados, así que **más es mejor** — con la reserva que mide B.3.

| Modo de normalización | Qué le hace al valor guardado **y** al pedido |
|---|---|
| `raw` | nada: compara tal cual está en el catálogo |
| `casefold` | pasa los dos a minúsculas |
| `unaccent` | minúsculas y además sin tildes |

La columna `ganancia_x` deja hecha la división: cuántas veces más alcanza *contiene* que *igualdad exacta* en esa misma fila.

In [3]:
# 📄 DATOS · 📚 catalogo_productos.csv (15.000)
# Los tres colores más frecuentes salen del propio catálogo, no de una lista
# escrita a mano: elegirlos a dedo sería elegir el resultado.
colores_frecuentes = (
    value_frequency(completo, "color")
    .query("color != '(vacío)'")
    .head(3)["color"].str.lower().tolist()
)

comparacion = pd.concat([
    filter_reach(completo, colores_frecuentes, field="color", match=match)
    for match in ("equals", "contains")
])
print(f"colores medidos: {colores_frecuentes}")

# Los nombres de columna del pivote son los de la API (`equals`/`contains`);
# se renombran solo para mostrar, que es donde los lee una persona.
igualdad_contiene = (
    comparacion
    .pivot_table(index=["filtro", "modo"], columns="match", values="n_productos")
    .rename(columns={"equals": "igualdad_exacta", "contains": "contiene"})
    .rename_axis(index={"filtro": "color_pedido", "modo": "normalizacion"})
)
igualdad_contiene.assign(
    ganancia_x=lambda d: (d["contiene"] / d["igualdad_exacta"]).round(1)
)

colores medidos: ['negro', 'blanco', 'multicolor']


match                       contiene  igualdad_exacta  ganancia_x
color_pedido normalizacion                                       
blanco       casefold         1115.0            731.0         1.5
             raw               104.0             56.0         1.9
             unaccent         1115.0            731.0         1.5
multicolor   casefold          514.0            420.0         1.2
             raw                54.0             50.0         1.1
             unaccent          514.0            420.0         1.2
negro        casefold         2213.0           1538.0         1.4
             raw               191.0            131.0         1.5
             unaccent         2213.0           1538.0         1.4

## B.2 · Cómo lo escribe el usuario

La tabla anterior compara políticas suponiendo **una sola forma** de escribir la consulta, la minúscula: una suposición cómoda y sesgada a favor de normalizar, porque nadie escribe siempre igual.

Aquí la suposición se sustituye por una medición: la misma consulta escrita como la escribiría una persona —todo minúsculas, inicial en mayúscula, todo mayúsculas y, si la palabra lleva tilde, las tres formas otra vez con ella— contra los tres modos de normalización.

### Cómo leer las tablas

Las **filas** son lo que escribe la persona. Las **columnas** son lo que hace el sistema con esa consulta **y con el valor guardado** antes de compararlos:

| Modo | Qué hace |
|---|---|
| `raw` | no toca nada |
| `casefold` | pasa ambos a minúsculas |
| `unaccent` | minúsculas y además quita tildes a ambos |

Lo que se busca es la columna donde **todas las filas dan el mismo número** y ese número es **el más alto** de la tabla.

El veredicto tiene **dos ejes que conviene no mezclar**:

| Eje | Qué pregunta | Por qué no basta el otro |
|---|---|---|
| **Consistente** | ¿Encuentran todos lo mismo? | Un filtro cuyo resultado depende de la tecla de mayúsculas no es un filtro |
| **Alcanza el máximo** | ¿Y encuentran *todo* lo que hay? | Que todo el mundo encuentre la mitad también es consistente, y está igual de roto |

La segunda columna existe porque el primer diseño solo medía la primera, y un test destapó el caso: quitar solo las mayúsculas deja a quien escribe la tilde y a quien no encontrando cada uno su mitad del catálogo, tan consistentes como incompletos.

In [ ]:
# 📄 DATOS · 📚 catalogo_productos.csv (15.000)
# Se prueban dos consultas: el color más frecuente y el más frecuente que
# lleva tilde. El segundo sale del catálogo, no de mi cabeza: sin una palabra
# acentuada, el eje de las tildes no se puede medir.
frecuencias = value_frequency(completo, "color").query("color != '(vacío)'")
con_tilde = frecuencias[
    frecuencias["color"] != frecuencias["color"].map(strip_accents)
]
consultas_escritura = [frecuencias["color"].iloc[0]]
if len(con_tilde):
    consultas_escritura.append(con_tilde["color"].iloc[0])

for consulta in consultas_escritura:
    tabla = filter_writing_robustness(
        completo, consulta, field="color", match="contains"
    )
    print(f"── {consulta!r} ──")
    display(tabla.pivot(index=["variante", "escritura"], columns="modo",
                        values="n_productos"))
    display(robustness_summary(tabla))

In [ ]:
# 📄 DATOS · 📚 catalogo_productos.csv (15.000)
# Lo mismo sobre la marca, que es el filtro obligatorio. Las cuatro marcas del
# enunciado se escriben con mayúsculas distintas entre sí -NIKE, Apple,
# SAMSUNG, Einhell-, así que aquí el eje de la caja no es hipotético.
for marca in marcas_pedidas:
    tabla = filter_writing_robustness(completo, marca, field="brand")
    resumen = robustness_summary(tabla).assign(marca=marca)
    display(resumen[["marca", "modo", "n_variantes", "alcances_distintos",
                     "minimo", "maximo", "consistente", "alcanza_el_maximo"]])

## B.3 · Lo que cuesta el *contiene*: falsos positivos

Buscar un texto dentro de otro alcanza más productos, pero también entra donde no debería: el texto buscado puede aparecer **dentro de otra palabra** en vez de como palabra suelta. Pedir `rosa` y que entre `rosado` es el caso típico.

### Cómo leer la tabla

| Columna | Qué cuenta | Dirección |
|---|---|---|
| `n_productos` | todo lo que alcanza el *contiene* | más es mejor |
| `n_dentro_de_otra_palabra` | de esos, los que entran **solo** porque el texto está dentro de otra palabra | ⚠️ **más es peor**: son falsos positivos |
| `pct_falsos_positivos` | qué parte del alcance es basura | ⚠️ más es peor |

La coincidencia exacta y los valores compuestos (`Negro/Rojo`, `Blanco (White)`) **no** cuentan como falso positivo: ahí el color pedido está como palabra suelta, que es lo que el *contiene* viene a rescatar.

Es lo que convierte la política en decisión medida: si el porcentaje es despreciable, el *contiene* sale gratis; si no, hay que decidir si el alcance extra compensa la pérdida de precisión.

> 📌 Se mide con la clave **normalizada**, no con las tres. El coste hay que verlo en la configuración que se va a desplegar: sobre el valor crudo el alcance es diez veces menor (B.4) y los falsos positivos saldrían artificialmente bajos.

### ⚠️ Limitación declarada: el género y el número

Hay un hueco que **ninguna política de filtro resuelve**, y conviene dejarlo escrito antes de que lo encuentre otro: `negro` y `negra` son cadenas distintas y **ninguna contiene a la otra**, así que no se encuentran entre sí ni por subcadena, ni por palabras, ni con igualdad.

| Filtro | Subcadena | Por palabras |
|---|---:|---:|
| `negro` | 215 | 212 |
| `negra` | 3 | 3 |

Los 3 de `negra` **no están** entre los 215 ni entre los 212. Y los 3 de diferencia entre 215 y 212 son otra cosa: valores donde `negro` aparece dentro de otra palabra — los falsos positivos que cuenta esta sección.

**La asimetría es lo grave.** Quien busca `negro` pierde 3 productos, un 0,2 %; quien busca `negra` recibe esos 3 y **pierde los otros 212**: la forma menos frecuente devuelve un resultado plausible y esconde el grueso del catálogo sin avisar. Un vacío ruidoso se detecta; una página con tres resultados correctos, no.

No depende del motor —la tienen los tres candidatos, así que **no entra en R03**— y las salidas serían *stemming* en el tokenizador o normalizar el género al construir `color_normalized`, que añaden un eje que habría que medir. Se declara como limitación conocida, que es lo que §5 llama *"atribución de errores: datos o filtros"*.

### Cuándo dejaría de importar todo esto

El requisito duro de esta sección —que el motor sepa buscar texto dentro de un metadato— y la ventaja del nivel 2 **solo pesan porque `color` es texto libre**. Con una taxonomía limpia —un color por registro, vocabulario cerrado y **ningún valor subcadena de otro**—, `like '%x%'` no podría dar falsos positivos y los tres motores empatarían en este eje.

Y esa condición no hay que juzgarla a ojo: **es la columna de arriba**.

> `n_dentro_de_otra_palabra == 0` para todos los colores filtrados ⟺ el `contains` por subcadena es seguro

Hoy no se cumple, y por partida doble:

- La sección B midió que `color` **no es una taxonomía** sino texto libre. La muestra trae `"2 unidades negra"`, `"Negro, 1 Piezas"`, `"negra + oro rosa"`, `"como se muestra"` — cantidades, recuentos de piezas y varios colores en el mismo campo.
- `oro` es subcadena de `incoloro`, que es el caso que los tests fijan.

Si el catálogo se limpiara —o si el color viniera de un desplegable en vez de texto libre— **R03 podría decidirse por los otros criterios**: memoria, control del ANN, calidad del error, dependencia del proveedor. Conviene reevaluarlo entonces en vez de arrastrar una restricción que ya no aplica.

In [ ]:
# 📄 DATOS · 📚 catalogo_productos.csv (15.000)
falsos = filter_reach(
    completo, colores_frecuentes, field="color",
    match="contains", modes=(NORMALIZACION,),
)
falsos.assign(
    pct_falsos_positivos=lambda d: (
        100 * d["n_dentro_de_otra_palabra"] / d["n_productos"]
    ).round(2)
)[["filtro", "n_productos", "n_dentro_de_otra_palabra",
   "pct_falsos_positivos", "n_valores_distintos", "valores"]]

## B.4 · La clave derivada, y por qué el filtro no puede vivir sin ella

**D03** decidió dos cosas que solo encajan si el punto lleva una clave más: el valor **se guarda tal cual viene** y **se normaliza al buscar**. Un filtro nativo compara lo almacenado con lo pedido byte a byte, así que normalizar solo la consulta no sirve: la normalización tiene que estar materializada en el punto. La misma política se extiende ahora al color.

### Cómo leer las dos tablas

Son las dos caras de la decisión: la primera dice **qué cuesta** la clave, la segunda **qué compra**.

**Coste** — `mb_total` de cada clave derivada, que se suma a lo que ya ocupa el valor crudo (D03 guarda los dos), y se compara con los MB de vectores de la sección A.

**Beneficio** — las dos columnas son lo mismo medido de dos formas, renombradas porque el nombre de la API despista:

| Columna | Qué es en realidad |
|---|---|
| `sin_clave_derivada` | el filtro compara contra el valor **tal cual está guardado**; solo la consulta se normalizó. Es lo que pasa si el punto no lleva la clave |
| `con_clave_derivada` | los dos lados normalizados, que es lo que D03 exige |
| `pct_alcanzado_sin_clave` | qué parte del alcance real consigue la primera. ⚠️ **cuanto más bajo, más se pierde** |

Ojo con el sentido: un `pct_alcanzado_sin_clave` del 9 % significa que sin la clave **se pierde el 91 %** de lo que debería encontrarse. Y no es un subconjunto cualquiera: son los productos que alguien tecleó con mayúscula al cargar el catálogo, o sea un filtro que falla en silencio y sin patrón.

In [ ]:
# 📄 DATOS · 📚 catalogo_productos.csv (15.000)
con_claves = completo.copy()
for campo in CAMPOS_FILTRABLES:
    con_claves = add_normalized_key(con_claves, field=campo, mode=NORMALIZACION)

# 1) Lo que CUESTA la clave: bytes por campo derivado.
derivadas = [f"{campo}_normalized" for campo in CAMPOS_FILTRABLES]
display(field_byte_profile(con_claves, derivadas))

# 2) Lo que COMPRA: el mismo filtro con y sin ella. `raw` no es "no normalizar
# nada", es "normalizar solo la consulta" -que es el escenario sin clave.
sin_clave = filter_reach(
    completo, colores_frecuentes, field="color",
    match="contains", modes=("raw", NORMALIZACION),
)
ganancia = (
    sin_clave
    .pivot(index="filtro", columns="modo", values="n_productos")
    .rename(columns={"raw": "sin_clave_derivada",
                     NORMALIZACION: "con_clave_derivada"})
    .rename_axis(index="color_pedido", columns=None)
)
ganancia.assign(
    pct_alcanzado_sin_clave=lambda d: (
        100 * d["sin_clave_derivada"] / d["con_clave_derivada"]
    ).round(1)
)

## B.5 · Filtros combinados: el escenario donde filtrar después se rompe

Una marca sola ya es selectiva. Una marca **con** un color concreto lo es mucho más, y ahí es donde recuperar diez vecinos y descartar los que no cumplen deja de funcionar: puede no quedar ninguno.

### Cómo leer la tabla

Cada fila es un cruce marca × color. La marca se compara con **igualdad** y el color con **contiene**, cada uno contra su clave normalizada — las mismas políticas que se acaban de decidir, no otras.

| Columna | Qué cuenta |
|---|---|
| `n_brand` | productos de esa marca |
| `n_brand_con_color` | de esos, cuántos tienen el color **anotado** |
| `n_brand_sin_color` | de esos, cuántos lo tienen vacío |
| `n_brand_y_color` | los que pasan **los dos** filtros |
| `pct_del_catalogo` | qué parte de los 15.000 sobrevive al cruce |
| `cero_por_falta_de_dato` | ⚠️ el cero de esa fila **no dice nada del catálogo** |

Las tres columnas del medio están porque **un cero sin ellas es ambiguo**: con un 37 % de colores vacíos, `n_brand_y_color = 0` puede significar que la marca no tiene ningún producto de ese color —un hecho del catálogo, y el argumento se sostiene— o que sus productos no lo tienen anotado —un hecho sobre la cobertura, que cambia en cuanto alguien rellene la columna—. `cero_por_falta_de_dato` marca el segundo caso para que no se lea como el primero.

Cuanto más baja sea `pct_del_catalogo`, más se refuerza el argumento: si el filtro deja menos supervivientes que los 10 resultados pedidos, recuperar primero y descartar después no puede llenar la página.

In [ ]:
# 📄 DATOS · 📚 catalogo_productos.csv (15.000)
# La función recibe las columnas SIN normalizar y normaliza ella los dos lados
# -lo pedido y lo almacenado- con la política de D03, igual que filter_reach.
# Repetir esa normalización a mano en la celda es como se coló antes un filtro
# de marca que comparaba contra el valor crudo.
combinados = combined_filter_selectivity(
    completo,
    primary_values=marcas_pedidas,
    secondary_values=colores_frecuentes,
    primary_field="brand",
    secondary_field="color",
    mode=NORMALIZACION,
)
combinados.sort_values("n_brand_y_color")

### Qué se lleva el esquema de todo esto

Tres consecuencias, y ninguna es opcional una vez tomada la decisión del campo extra:

1. **El payload lleva cuatro claves donde parecía llevar dos:** el valor crudo de marca y color —que es lo que se enseña al usuario— y su versión normalizada —que es contra lo que filtra el motor—. Sin las derivadas, el filtro no puede cumplir D03.
2. **El motor tiene que saber buscar texto dentro de un campo de metadatos.** Es un requisito duro: quien no pueda queda descartado sin prueba de humo. Filtra la lista de candidatos y por eso está escrito aquí, antes de mirarlos.
3. **Hay que indexar los dos campos derivados**, no solo uno, y ese coste se paga en la ingesta.

> ⚠️ Lo obligatorio manda sobre la mejora: las cuatro consultas del enunciado filtran por marca con igualdad, y eso tiene que seguir siendo exacto. El color se añade **al lado**, nunca cambiando cómo se comporta la marca.

---

# C · D13 · Qué se guarda en cada punto

El enunciado (§3.2) exige definir *"esquema, dimensión, métrica, IDs, metadatos y política de valores nulos"*. Esta sección cubre los metadatos; la D, los nulos.

**Buena parte de esta decisión no se decide: se hereda.** Los notebooks posteriores consumen campos concretos, y quitarlos no es ahorrar sino romper lo que viene después:

| Campo | Quién lo necesita |
|---|---|
| `product_id` | Los CSV de salida del §6 |
| `title` | NB05 para mostrar el resultado (§3.3 lo exige) y NB07 para la similitud de título |
| `brand`, `color` | Lo que se le enseña al usuario |
| `brand_normalized`, `color_normalized` | Contra lo que filtra el motor — sección B |
| `catalog_version`, `active` | NB08, para comprobar la versión y la baja |
| `text` | **Nadie** |

Con eso, `minimo` queda descartado por los requisitos de aguas abajo y no por preferencia. **Lo que D13 decide de verdad es el último escalón:** si el índice guarda además el texto de origen.

### La pregunta, en una línea

> ¿El índice debe poder reconstruirse **sin el CSV**?

Guardar `text` hace el índice autosuficiente. Cuesta lo que cuesta el campo más largo del catálogo, y es el único que ningún notebook posterior abre.

### Cómo leer la tabla

Una fila por esquema, ordenados de menos a más campos y **anidados** —cada uno contiene al anterior—, para que la comparación mida *qué añade llevar más* y no dos esquemas distintos.

| Columna | Qué es | Dirección |
|---|---|---|
| `bytes_medios` | lo que ocupa un payload típico | menos es mejor |
| `bytes_p95` · `bytes_max` | el caso malo y el peor | avisan de si la media engaña |
| `mb_total` | los 15.000 puntos juntos | se compara con los MB de vectores de la sección A |

El número que decide no es `mb_total` a secas sino **qué fracción de la memoria del índice representa**: un payload que suma un 2 % sobre los vectores es ruido; uno que suma un 30 % es una decisión.

In [ ]:
# 📄 DATOS · 📚 catalogo_productos.csv (15.000)
# Los tres esquemas de D13 salen de PAYLOAD_SCHEMAS, no de una lista escrita
# en la celda: así el notebook no puede inventarse un cuarto a mitad de tabla.
print("Esquemas candidatos:")
for nombre, campos in PAYLOAD_SCHEMAS.items():
    print(f"  {nombre:<18} {len(campos)} campos · {', '.join(campos)}")

# Se presupuesta sobre el catálogo CON las claves derivadas ya calculadas:
# son parte del punto que se va a escribir, no un extra.
presupuesto = payload_budget(con_claves, null_policy="omitir_campo")
presupuesto.assign(
    pct_sobre_vectores=lambda d: (100 * d["mb_total"] / huella["mb_vectores"]).round(1)
)

### C.1 · De dónde sale cada byte

La tabla anterior dice cuánto ocupa cada esquema; esta, **qué campo se lo come**. Si un solo campo explica casi todo el salto, la elección deja de ser "mínimo o completo" y pasa a ser "ese campo, sí o no".

In [ ]:
# 📄 DATOS · 📚 catalogo_productos.csv (15.000)
campos_del_maximo = list(PAYLOAD_SCHEMAS["completo_con_text"])
por_campo = field_byte_profile(con_claves, campos_del_maximo)
por_campo.assign(
    pct_del_payload=lambda d: (100 * d["mb_total"] / d["mb_total"].sum()).round(1)
).sort_values("mb_total", ascending=False)

---

# D · D14 · Qué se escribe cuando el campo está vacío

**La decisión de más consecuencia de este notebook**, y la que menos lo parece. Afecta al **4,39 % de las marcas** y al **37,39 % de los colores** — más de cinco mil productos.

El motivo cabe en una frase: **`campo ausente` y `campo == ""` no son lo mismo para un motor**, aunque en Python lo parezcan. De ahí depende que un producto sin marca sea alcanzable o invisible, y eso cae directo sobre los filtros de NB05.

| Opción | Qué escribe en el punto | Qué le pasa al producto sin marca |
|---|---|---|
| `omitir_campo` | la clave no existe | **Invisible** a cualquier filtro sobre ese campo. En varios motores ni siquiera se puede preguntar "¿cuáles no tienen marca?" |
| `cadena_vacia` | `brand: ""` | Existe para el filtro; alcanzable con una condición explícita de vacío |
| `centinela` | `brand: "(desconocido)"` | Alcanzable y legible, pero **contamina**: un *contiene* que caiga dentro del centinela lo trae sin querer |

> 📌 **No confundir con D02**, que era la política de nulos en el *texto codificado*. Datos distintos, decisión distinta — y D02 además quedó sin efecto al ganar A4.

### La pregunta de negocio que hay debajo

Para las cuatro consultas del enunciado da igual: piden marcas concretas y los tres comportamientos coinciden. Importa para el **control de catálogo**: si nadie puede listar los productos sin marca, nadie los va a arreglar nunca.

### Cómo leer la tabla

Una fila por política sobre el mismo esquema, para que la única diferencia sea la política. `mb_total` es lo que cuesta; `n_puntos_afectados`, a cuántos les cambia el contenido del punto.

In [ ]:
# 📄 DATOS · 📚 catalogo_productos.csv (15.000)
# A cuántos puntos afecta la decisión, antes de mirar lo que cuesta: si
# fueran cuatro productos, la política sería indiferente.
display(field_byte_profile(con_claves, CAMPOS_FILTRABLES + derivadas)
        [["campo", "n_vacios", "pct_vacios"]])

# Las tres políticas sobre el MISMO esquema: lo único que cambia es qué se
# escribe en el hueco.
coste_nulos = pd.concat([
    payload_budget(con_claves, null_policy=politica,
                   schemas={"completo": PAYLOAD_SCHEMAS["completo"]})
    .assign(politica=politica)
    for politica in NULL_POLICIES
])
coste_nulos[["politica", "bytes_medios", "bytes_p95", "bytes_max", "mb_total"]]

### D.1 · El punto que se escribe, con cada política

Los números de arriba ordenan las opciones por coste; esta celda enseña **la consecuencia**: el mismo producto sin color escrito de las tres formas, que es lo que el motor recibe y lo que su filtro compara.

In [ ]:
# 📄 DATOS · 📚 catalogo_productos.csv (15.000)
# Un producto real sin color, para no razonar sobre un ejemplo inventado.
sin_color = con_claves[con_claves["color"].isna()].iloc[0]
print(f"producto {sin_color['product_id']} — color ausente en el origen\n")

for politica in NULL_POLICIES:
    punto = build_payload(
        sin_color, fields=PAYLOAD_SCHEMAS["completo"], null_policy=politica
    )
    presente = "color" in punto
    print(f"{politica:<14} clave presente: {presente!s:<5} "
          f"valor: {punto.get('color', '(sin clave)')!r}")

---

# E · D15 · Tamaño del lote de ingesta

⚠️ **No es la decisión de memoria que parecía.** Se planteó como *"con 8 GB y el modelo cargado, mide RAM antes de subir"*, pero aquí **el modelo no se carga**: los vectores vienen ya calculados de `artifacts/embeddings/`. A 768 dimensiones y 4 bytes por decimal, un vector ocupa 3.072 bytes y el lote más grande ronda el megabyte, así que la RAM no arbitra.

Lo que sí decide son tres cosas, y la celda las mide:

| Columna | Qué decide | Dirección |
|---|---|---|
| `pct_del_limite` · `cabe_en_un_mensaje` | El **único techo duro**: pasarse del máximo de mensaje gRPC no ralentiza, corta la petición | ⚠️ acercarse a 100 es fallo, no lentitud |
| `n_lotes` | Viajes de red. Menos lotes, menos ida y vuelta | menos es más rápido |
| `puntos_reintentados_si_falla` | Trabajo perdido cuando un lote se cae | menos es más seguro |

Los dos últimos tiran en direcciones opuestas: **el lote grande ahorra viajes y paga más cuando falla**. Ese es el compromiso real de D15.

> 🔗 **D13 y D15 están acopladas.** El techo de mensaje sube con el payload: si D13 se lleva `text`, el lote grande se acerca al límite. Por eso la celda se calcula con el esquema elegido y no con uno fijo.

In [ ]:
# 📄 DATOS · 📚 catalogo_productos.csv (15.000)
# El payload medio sale del esquema de D13, no de un número redondo: cambiar
# de esquema tiene que mover esta tabla, que es de lo que avisa el acople.
ESQUEMA_D13 = "completo"      # ← al cambiarlo se ve el efecto sobre el lote

bytes_payload = float(
    payload_budget(con_claves, null_policy="omitir_campo",
                   schemas={ESQUEMA_D13: PAYLOAD_SCHEMAS[ESQUEMA_D13]})
    ["bytes_medios"].iloc[0]
)
print(f"esquema {ESQUEMA_D13}: {bytes_payload:.0f} bytes de payload por punto")
print(f"vector           : {DIM * 4} bytes por punto\n")

batch_footprint(DIM, n_points=len(completo), payload_bytes_medios=bytes_payload)

### Qué se lleva NB05 de estas tres decisiones

Las tres se pagan en el índice que se construye una vez, pero se cobran en cada consulta:

1. **D13** fija qué puede devolver `buscar()` sin volver al CSV. Un campo que no esté en el payload no está en el resultado, y §3.3 exige `product_id`, posición, título, metadatos y score.
2. **D14** fija si un producto con el metadato vacío es alcanzable. Es la que decide qué contesta el sistema a un filtro que nadie pensó al indexar.
3. **D15** solo se nota en el tiempo de ingesta y en qué pasa cuando algo falla a medio camino — que es exactamente lo que el paso 3 del guion de humo comprueba al repetirla.

---

# F · La prueba de humo — el mismo guion contra cada candidato

**D12** dejó tres motores en pie: **Qdrant**, **Weaviate** y **Milvus**. Los tres pasan por el mismo guion de diez pasos, escrito **una sola vez** en `aurum.motores.humo` y ejecutado contra un puerto común. Eso es lo que hace que la comparativa compare motores: si cada uno tuviera su guion, las diferencias de la tabla podrían venir del código.

### El corpus es la 🔬 muestra, no el catálogo

Al revés que el resto de NB04: aquí se prueba el recorrido completo —crear, ingerir, repetir, buscar, filtrar, leer, borrar— tres veces seguidas, y es donde conviene equivocarse barato. El índice bueno se construye después, una sola vez y con el ganador.

### Qué automatiza esta sección y qué no

| Pasos | Quién |
|---|---|
| **1–7** · crear · ingerir · repetir · buscar · filtrar · leer · borrar | Las celdas de abajo |
| **8–10** · reinicio · motor caído · RAM y volumen | **A mano**, desde la terminal |

Los tres últimos salen del proceso de Python, así que aparecen en la tabla como filas `✍️ manual` — presentes y vacías. Si no estuvieran, la persistencia, que es requisito del enunciado, desaparecería del artefacto sin que se note.

> ⚠️ **Antes de ejecutar nada, levanta el motor.** Los tres no caben a la vez en esta máquina: uno cada vez, y Milvus el último porque son tres contenedores.
>
> ```bash
> make motor-up MOTOR=qdrant     # espera al healthcheck
> ```

In [4]:
# 📄 DATOS · 🔬 catalogo_muestra.csv (1.500) + sus vectores A4 ya codificados
# Los vectores salen de la caché de NB03; no se recodifica nada aquí.
import os

import numpy as np
from dotenv import load_dotenv

from aurum.embeddings import GeminiEncoder, encode_corpus, truncate_dim, vector_health
from aurum.motores import (
    MANUAL_STEP_NUMBERS,
    MEDICION_ADVERTENCIAS,
    FilterCondition,
    Point,
    FilterProbe,
    contains_level,
    error_quality,     # paso 9: de quién es la excepción, no solo si la hay
    load_smoke,
    persistence_check, # paso 8: recuento, conjunto y orden por separado
    probe_filters,
    read_snapshot,
    record_manual,     # los pasos manuales se anotan, no se editan a mano
    resource_note,
    resource_table,    # paso 10: leído del transcrito de la terminal
    run_smoke_test,
    save_smoke,
    smoke_differences, # lo que la tabla de ✅/❌ esconde
    smoke_display,   # para mirar: envuelve el texto en vez de recortarlo
    smoke_table,     # para el artefacto: los mismos datos, sin formato
)
from aurum.plantillas import render_template

load_dotenv(Path("..") / ".env")
CACHE = Path("..") / "artifacts" / "embeddings"
MODELO, CONTRATO = "gemini-embedding-2", "sin_contrato"   # R02
PLANTILLA = "A4"                                          # R01
LOTE = 128                    # D15
ESQUEMA = "completo"          # D13
POLITICA_NULOS = "cadena_vacia"   # D14

textos = render_template(muestra, PLANTILLA)
codificado = encode_corpus(
    GeminiEncoder(api_key=os.environ.get("GEMINI_API_KEY"), model_id=MODELO,
                  native_dim=3072, window=8192),
    textos, corpus_id=f"catalogo_muestra__{PLANTILLA}",
    kind="document", contract=CONTRATO, batch_size=32, cache_dir=CACHE,
)
# 3.072 -> 768 y renormalizar: truncar un vector unitario deja la norma < 1
# y el coseno dejaría de ser un coseno.
vectores = truncate_dim(codificado.vectors, DIM)
# `stats.desde_cache` y no `metadata[...]`: el metadata guarda el valor de la
# codificación ORIGINAL -siempre False- y diría que se recodificó cuando no.
# Aquí eso no es cosmético: si sale False de verdad, se ha pagado a la API.
print(f"vectores: {vectores.shape} · desde caché: {codificado.stats.desde_cache}")
print(f"salud   : {vector_health(vectores)}")

c:\Users\asus\Master\modulos\modulo10_bbdd\practica\AURUM_MARKET\aurum-market-catalog\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


vectores: (1500, 768) · desde caché: True
salud   : {'n_vectores': 1500, 'dim': 768, 'dtype': 'float32', 'finito': True, 'norma_min': 1.0, 'norma_max': 1.0, 'normalizado': True, 'n_filas_duplicadas': 0, 'bytes_por_vector': 3072}


## F.1 · Los puntos, con el esquema que acaban de decidir C, D y E

Aquí se junta todo lo anterior: el vector de R01+R02, el payload de **D13**, los huecos de **D14** y las claves derivadas que la sección B hizo obligatorias. El identificador es `record_id` —UUIDv5, lo impone `README_DATOS`— y es lo que hace la idempotencia del paso 3 gratis: reingerir el mismo punto lo sobrescribe en vez de duplicarlo.

In [5]:
# 📄 DATOS · 🔬 catalogo_muestra.csv (1.500)
muestra_con_claves = muestra.copy()
for campo in CAMPOS_FILTRABLES:
    muestra_con_claves = add_normalized_key(
        muestra_con_claves, field=campo, mode=NORMALIZACION
    )

puntos = [
    Point(
        record_id=fila["record_id"],
        vector=vectores[i],
        payload=build_payload(
            fila, fields=PAYLOAD_SCHEMAS[ESQUEMA], null_policy=POLITICA_NULOS
        ),
    )
    for i, fila in enumerate(muestra_con_claves.to_dict("records"))
]

print(f"{len(puntos)} puntos · {len(set(p.record_id for p in puntos))} ids distintos")
print(f"payload de ejemplo: {puntos[0].payload}")

1500 puntos · 1500 ids distintos
payload de ejemplo: {'product_id': 'B0818K237B', 'title': 'Kanlin1986 Vestido Largo De Navidad para Mujer, Vestido Talla Grande Elegante De Manga Larga con Escote En La Manga De Navidad, Vestido Largo De Noche De Fiesta De Noche De Playa', 'brand': 'KanLin1986-Ropa', 'color': 'Negro', 'catalog_version': 1, 'active': True, 'brand_normalized': 'kanlin1986-ropa', 'color_normalized': 'negro'}


## F.2 · La consulta del paso 5, y por qué esa y no otra

El paso 5 comprueba que **el motor** ejecute el filtro auditando que todo lo devuelto cumpla la condición, y esa auditoría solo destapa a quien lo ignore **si la condición es selectiva**: con una marca que ya domine el top-10, un filtro ignorado pasaría por bueno.

De ahí **FILTER-001 del enunciado**: `"herramienta inalámbrica para perforar"` con `brand = Einhell`, el 0,2 % del catálogo. El detalle que lo hace la prueba correcta es que **la palabra "Einhell" no aparece en el texto de la consulta**: la búsqueda vectorial no tiene forma de preferir esa marca, así que si los resultados la cumplen es porque el filtro trabajó.

In [6]:
# 📄 DATOS · consultas_filtradas.csv (4) — se codifican aquí por primera vez
# NB02 y NB03 codificaron desarrollo y evaluación, no estas. Son 4 textos
# cortos: la caché hace que solo se pague una vez.
consulta = filtradas.iloc[0]
vec_consultas = encode_corpus(
    GeminiEncoder(api_key=os.environ.get("GEMINI_API_KEY"), model_id=MODELO,
                  native_dim=3072, window=8192),
    filtradas["query_text"].tolist(), corpus_id="consultas_filtradas",
    kind="query", contract=CONTRATO, batch_size=8, cache_dir=CACHE,
)
# Las CUATRO, no solo la primera: el paso 5 del guion usa una, pero la tabla
# de filtros las necesita todas -§5 las pide como evidencia mínima-.
vectores_consultas = truncate_dim(vec_consultas.vectors, DIM)
vector_consulta = vectores_consultas[0]      # FILTER-001, la del paso 5

# El filtro se expresa en los términos de la decisión -campo, valor, política-,
# no en el lenguaje de ningún motor: cada adaptador lo traduce al suyo, y esa
# traducción es justo donde se ve qué sabe hacer cada uno.
FILTRO = [FilterCondition(
    field="brand",
    value=normalize_brand(consulta["filter_value"], NORMALIZACION),
    operator="equals",   # D03: vocabulario cerrado, una marca por ficha
)]
print(f"{consulta['workload_id']}: {consulta['query_text']!r}")
print(f"filtro: {FILTRO[0].field} {FILTRO[0].operator} {FILTRO[0].value!r}")
print(f"⚠️ la marca NO aparece en el texto: "
      f"{consulta['filter_value'].lower() not in consulta['query_text'].lower()}")

FILTER-001: 'herramienta inalámbrica para perforar'
filtro: brand equals 'einhell'
⚠️ la marca NO aparece en el texto: True


## F.3 · Ejecutar el guion — ⏱️ **una celda por motor**

Cambia `MOTOR` y ejecuta. Antes de cada uno: levanta ese y **baja el anterior**.

| Motor | Levantar | Puertos |
|---|---|---|
| `qdrant` | `make motor-up MOTOR=qdrant` | 6333 · 6334 |
| `weaviate` | `make motor-up MOTOR=weaviate` | 8080 · 50051 |
| `milvus` | `make motor-up MOTOR=milvus` | 19530 · 9091 |

> 🔒 La colección se borra y se recrea en cada pasada, así que necesita `AURUM_ALLOW_RESET=true` en el `.env`. Son dos barreras a propósito —el nombre tiene que empezar por `aurum_humo` **y** el permiso tiene que estar dado—, y conviene devolverlo a `false` al terminar NB04.

### Cómo leer la tabla

Una fila por paso. `esperado` es lo que el guion exige; `observado`, lo que hizo el motor.

| `resultado` | Significa |
|---|---|
| ✅ pasa | El motor cumple ese paso |
| ❌ falla | No cumple — y la fila dice qué pasó, incluida la excepción si la hubo |
| ✍️ manual | Se rellena a mano tras ejecutar el comando de F.4 |

**Un paso que falla no interrumpe el guion.** Un motor que no sepa filtrar tiene que llegar igualmente al paso de borrado: la tabla vale para comparar porque todas las filas están rellenas, y parar en el primer fallo dejaría al motor peor *descrito*, no peor *valorado*.

In [7]:
# Andamiaje común: se ejecuta una vez y sirve para los tres motores.
CANDIDATOS = ("qdrant", "weaviate", "milvus")        # D12
HUMO_DIR = Path("..") / "artifacts" / "humo"
SDK = {"qdrant": "qdrant_client", "weaviate": "weaviate", "milvus": "pymilvus"}

# Reentrante a propósito. `HUMO = {}` borraría una pasada que estuviera en
# memoria, y volver a medirla cuesta levantar el motor otra vez. Manda lo que
# haya en el kernel; si no hay nada, lo que se guardó en disco; si tampoco,
# se empieza vacío.
_en_disco, _filtros_en_disco = load_smoke(HUMO_DIR)
HUMO = globals().get("HUMO") or _en_disco
FILTROS = globals().get("FILTROS") or _filtros_en_disco
PERSISTENCIA = globals().get("PERSISTENCIA") or {}


def guardar(motor):
    """Deja en disco lo medido contra ese motor, en cuanto se mide.

    Sin esto el resultado vive solo en la memoria del kernel y reescribir la
    comparativa exige volver a levantar los tres motores. El §8 pide lo
    contrario: que los artefactos se regeneren sin repetir la medición."""
    save_smoke(HUMO_DIR, motor=motor, results=HUMO[motor], filters=FILTROS.get(motor))


def anotar(motor, paso, observado, pasa):
    """Rellena una de las tres filas manuales y la deja guardada.

    La fila ya trae escrito su `esperado` desde antes de medir, así que lo
    observado se apunta CONTRA el criterio y no en lugar de él."""
    record_manual(HUMO[motor], paso, observed=observado, passed=pasa)
    guardar(motor)
    print(f"  → fila {paso} de {motor}: {'✅ pasa' if pasa else '❌ falla'}")


def abrir(motor):
    """Construye el adaptador del motor. El SDK se importa aquí dentro: probar
    uno no obliga a tener instalados los otros dos."""
    coleccion = f"aurum_humo_{motor}"   # el prefijo es la primera salvaguarda
    if motor == "qdrant":
        from aurum.motores.qdrant import QdrantStore
        return QdrantStore(
            collection=coleccion,
            url=os.environ.get("AURUM_QDRANT_URL", "http://localhost:6333"),
            api_key=os.environ.get("AURUM_QDRANT_API_KEY"),
        )
    if motor == "weaviate":
        from aurum.motores.weaviate import WeaviateStore
        return WeaviateStore(
            collection=coleccion,
            host=os.environ.get("AURUM_WEAVIATE_HOST", "localhost"),
            api_key=os.environ.get("AURUM_WEAVIATE_API_KEY"),
        )
    if motor == "milvus":
        from aurum.motores.milvus import MilvusStore
        return MilvusStore(
            collection=coleccion,
            uri=os.environ.get("AURUM_MILVUS_URI", "http://localhost:19530"),
            token=os.environ.get("AURUM_MILVUS_TOKEN"),
        )
    raise ValueError(f"D12 dejó tres motores; {motor!r} no es uno")


def probar(motor):
    """Ejecuta los pasos 1-7 contra un motor y guarda el resultado.

    Conserva las anotaciones manuales que ya tuviera ese motor: los pasos 9 y
    10 son propiedades del motor y del despliegue, no de esta coleccion, y
    perderlos por reingerir costaria volver a pararlo y volver a medirlo. El
    8 si depende de la coleccion, y por eso se avisa de que hay que rehacerlo.
    """
    previos = {
        r.step: r for r in HUMO.get(motor, [])
        if r.step in MANUAL_STEP_NUMBERS and r.passed is not None
    }
    store = abrir(motor)
    try:
        print(f"{motor} · versión del servidor: {store.server_version()}")
        HUMO[motor] = run_smoke_test(
            store, puntos, query_vector=vector_consulta, dim=DIM,
            metric="cosine", top_k=TOP_K, batch_size=LOTE, filters=FILTRO,
        )
        for paso, anterior in previos.items():
            record_manual(HUMO[motor], paso,
                          observed=anterior.observed, passed=anterior.passed)
        if previos:
            print(f"anotaciones manuales conservadas: {sorted(previos)}"
                  + (" — el paso 8 hay que rehacerlo, la coleccion es otra"
                     if 8 in previos else ""))
    finally:
        # Siempre, incluso si el guion revienta: con 7,9 GB no se puede dejar
        # una conexión abierta antes de levantar el siguiente motor.
        store.close()
    guardar(motor)   # antes de mostrar nada: la pasada ya no depende del kernel
    # `smoke_display` y no `smoke_table`: pandas recorta las celdas largas con
    # "..." y aquí lo largo es lo que hay que leer -el mensaje de la excepción
    # cuando un paso falla-. Los datos son los mismos; cambia cómo se muestran.
    return smoke_display(HUMO[motor], motor=motor)


# ── Las sondas de filtro: lo que el paso 5 del guion NO llega a probar ───
# El guion ejerce UNA consulta con igualdad. Aquí van las cuatro del
# enunciado (§5 las pide como evidencia mínima) y el `contains` sobre color,
# que la sección B declaró requisito duro y que ningún paso ejerce.
COLOR_SONDA = colores_frecuentes[0]          # el más frecuente del catálogo
COLOR_FRAGMENTO = COLOR_SONDA[:-1]           # "negro" -> "negr"


def oraculo(valor, *, campo, match):
    """Cuántos productos cumplen la condición SEGÚN EL CATÁLOGO, en pandas.

    Sin esto un cero del motor no dice nada: puede ser que esa marca no tenga
    productos en los 1.500 de la muestra, o que el filtro esté roto, y las dos
    cosas se leen igual. Es la misma idea que el enunciado pide para la
    fidelidad ANN -comparar contra un oráculo exacto-, aplicada al filtro.

    Se calcula sobre `muestra`, que es EXACTAMENTE el corpus ingerido: usar el
    catálogo completo daría un número que el motor no puede alcanzar.
    """
    fila = filter_reach(
        muestra, [valor], field=campo, match=match, modes=(NORMALIZACION,)
    ).iloc[0]
    return int(fila["n_productos"])


SONDAS = [
    FilterProbe(
        name=fila["workload_id"],
        query_vector=vectores_consultas[i],
        conditions=[FilterCondition(
            "brand", normalize_brand(fila["filter_value"], NORMALIZACION), "equals"
        )],
        role="obligatorio",
        expected=oraculo(fila["filter_value"], campo="brand", match="equals"),
        note=f'{fila["query_text"]!r} — la marca no aparece en el texto',
    )
    for i, fila in enumerate(filtradas.to_dict("records"))
] + [
    FilterProbe("color · igualdad", vector_consulta,
                [FilterCondition("color", COLOR_SONDA, "equals")],
                role="referencia",
                expected=oraculo(COLOR_SONDA, campo="color", match="equals"),
                note="referencia: cuántos casan con el valor entero"),
    FilterProbe("color · contiene palabra", vector_consulta,
                [FilterCondition("color", COLOR_SONDA, "contains")],
                role="palabra",
                expected=oraculo(COLOR_SONDA, campo="color", match="contains"),
                note="si supera a la igualdad, alcanza los compuestos de B.1"),
    FilterProbe("color · contiene fragmento", vector_consulta,
                [FilterCondition("color", COLOR_FRAGMENTO, "contains")],
                role="fragmento",
                expected=oraculo(COLOR_FRAGMENTO, campo="color", match="contains"),
                note="0 ⇒ por palabras (nivel 2) · >0 ⇒ subcadena literal (nivel 3)"),
]

print("oráculo (sobre los 1.500 ingeridos, sin motor de por medio):")
for s in SONDAS:
    print(f"  {s.name:<28} {s.conditions[0].field} {s.conditions[0].operator}"
          f" {s.conditions[0].value!r:<12} → {s.expected} productos")


def probar_filtros(motor):
    """Las 4 consultas del enunciado + las 3 sondas de `contains`, de una vez."""
    store = abrir(motor)
    try:
        # El paso 7 del guion borró un punto para comprobar el borrado, así
        # que la colección tiene 1.499 y el oráculo cuenta sobre 1.500. Si el
        # punto borrado casara con alguna sonda, la fila diría "devuelve de
        # menos" -o FILTRO ROTO si era el único- siendo el motor correcto.
        # Con Einhell a 1 solo producto en la muestra, ese riesgo es real.
        # Se repone: el upsert es idempotente por record_id, así que si no
        # faltaba no cambia nada.
        store.upsert(puntos[:1], batch_size=1)
        n = store.count()
        if n != len(puntos):
            print(f"⚠️ la colección tiene {n} puntos y el oráculo cuenta sobre "
                  f"{len(puntos)}: los veredictos van a salir sesgados")
        FILTROS[motor] = probe_filters(store, SONDAS, top_k=TOP_K)
    finally:
        store.close()
    if motor in HUMO:
        guardar(motor)

    tabla = FILTROS[motor]
    obligatorias = tabla[tabla["papel"] == "obligatorio"]
    # §8: "las consultas filtradas nunca devuelven otra marca". Con cero
    # resultados la condición se cumple de forma vacía, así que el oráculo
    # decide si ese cero es una ausencia real o un filtro roto.
    con_resultados = obligatorias[obligatorias["n_resultados"] > 0]
    rotos = obligatorias[obligatorias["veredicto"].str.contains("ROTO")]
    print(f"{motor} · marca: {int(con_resultados['todos_cumplen'].sum())}"
          f"/{len(con_resultados)} consultas con resultados, todas de la marca pedida")
    print(f"{motor} · ceros : {len(obligatorias) - len(con_resultados)} sin resultados"
          f" — de ellos {len(rotos)} son FILTRO ROTO según el catálogo")
    print(f"{motor} · contains: {contains_level(tabla)}")
    return tabla.style.set_properties(**{
        "white-space": "pre-wrap", "text-align": "left", "vertical-align": "top",
    }).hide(axis="index")


def instantanea(motor, etiqueta):
    """PASO 8 · Solo lee: no crea, no ingiere, no borra.

    Llamar a `probar()` otra vez NO sirve para esto: recrea la colección y
    borraría justo los datos cuya supervivencia se quiere comprobar.

    Con las dos mitades hechas, `persistence_check` separa las tres preguntas
    que un `==` entre listas junta en un solo booleano: recuento, conjunto de
    ids y orden. Solo las dos primeras son persistencia.
    """
    store = abrir(motor)
    try:
        PERSISTENCIA[(motor, etiqueta)] = read_snapshot(
            store, vector_consulta, top_k=TOP_K
        )
    finally:
        store.close()

    actual = PERSISTENCIA[(motor, etiqueta)]
    print(f"{motor} · {etiqueta}: count={actual.count} · top-{len(actual.ids)} leído")
    if etiqueta not in ("antes", "despues"):
        return   # lectura suelta de F.3e; se compara a mano con `comparar()`
    antes, despues = (PERSISTENCIA.get((motor, e)) for e in ("antes", "despues"))
    if not (antes and despues):
        print("  (falta la otra mitad: reinicia el motor y llama con la otra etiqueta)")
        return

    check = persistence_check(antes, despues)
    print(f"  recuento : {antes.count} → {despues.count}")
    print("  conjunto : " + ("los mismos ids" if check.same_set else
                             f"salen {len(check.lost)}, entran {len(check.gained)}"
                             f" (solapamiento {check.overlap:.0%})"))
    print("  orden    : " + ("igual" if check.same_order else
                             f"{check.moved} posiciones cambian"))
    if check.max_score_shift is not None:
        # La prueba de que un reordenamiento es desempate y no otro índice.
        print(f"  score    : se mueve como mucho {check.max_score_shift:.2e}")
    print(f"\n  {check.verdict()}")
    if motor in HUMO:
        anotar(motor, 8, check.verdict(), check.passed)


def comparar(motor, etiqueta_a, etiqueta_b):
    """Dos instantáneas cualesquiera, sin anotar nada. Diagnóstico de F.3e.

    `instantanea` compara «antes» contra «despues» porque ese par ES el paso
    8. Aquí hacen falta tres lecturas para separar dos causas que ese par
    confunde: el tiempo que pasa y el reinicio en sí."""
    check = persistence_check(PERSISTENCIA[(motor, etiqueta_a)],
                              PERSISTENCIA[(motor, etiqueta_b)])
    print(f"{motor} · {etiqueta_a} → {etiqueta_b}")
    print(f"  {check.verdict()}")
    return check


def probar_caido(motor):
    """PASO 9 · Ejecutar SOLO con el motor parado. Con él vivo no prueba nada.

    No mide que el motor se caiga -se para a propósito-, sino qué cuenta el
    SDK cuando pasa. Y distingue tres cosas que la palabra "tipada"
    junta: una excepción de `builtins` obliga a adivinar por el mensaje, una
    del SDK se captura por tipo, y una del transporte (gRPC) es tipada pero
    ata el manejo de errores de NB05 a la capa de red del cliente.
    """
    try:
        caido = abrir(motor)
        caido.search(vector_consulta, top_k=TOP_K)
    except Exception as error:
        observado, pasa = error_quality(error, sdk_package=SDK[motor])
        print(observado)
        if motor in HUMO:
            anotar(motor, 9, observado, pasa)
    else:
        print("El motor sigue respondiendo: no está parado, así que el paso 9 no prueba nada")


# Lo que ya estuviera medido, a disco. Aquí y no al final: si esta celda se
# ejecuta con una pasada en memoria que nunca se guardó, esa pasada deja de
# depender del kernel en el mismo momento.
for _motor in list(HUMO):
    guardar(_motor)
print(f"medido: {sorted(HUMO) or 'nada todavía'} · guardado en {HUMO_DIR}")
print("listo · ejecuta el bloque del motor que tengas levantado")

oráculo (sobre los 1.500 ingeridos, sin motor de por medio):
  FILTER-001                   brand equals 'einhell'    → 1 productos
  FILTER-002                   brand equals 'apple'      → 0 productos
  FILTER-003                   brand equals 'nike'       → 6 productos
  FILTER-004                   brand equals 'samsung'    → 5 productos
  color · igualdad             color equals 'negro'      → 164 productos
  color · contiene palabra     color contains 'negro'      → 215 productos
  color · contiene fragmento   color contains 'negr'       → 218 productos
medido: ['milvus', 'qdrant', 'weaviate'] · guardado en ..\artifacts\humo
listo · ejecuta el bloque del motor que tengas levantado


### F.3a · Qdrant

El nº 1 del orden de preferencia previo (D12). Panel web en <http://localhost:6333/dashboard>, sin clave.

Lo que hay que mirar con atención en su fila: **el paso 5**. Qdrant resuelve el `contains` con `MatchText` sobre un índice de texto declarado al crear la colección, que es coincidencia **por palabras** y no subcadena literal. Si funciona, da el alcance de B.1 **sin** los falsos positivos de B.3 — una ventaja que no estaba prevista.

#### ① Levantar

```bash
make motor-up MOTOR=qdrant
```

#### ② Pasos 1–7 · automáticos

In [ ]:
probar("qdrant")

qdrant · versión del servidor: 1.18.2


motor,paso,comprobacion,que_ha_hecho,esperado,observado,resultado,segundos
qdrant,1,Crear colección,"create_collection(dim=768, metric='cosine', recreate=True)","dim=768, métrica=cosine","creada (dim=768, métrica=cosine)",✅ pasa,1.156000
qdrant,2,Ingesta por lotes,"upsert(1500 puntos, batch_size=128) → count()",count() == 1500,count() = 1500,✅ pasa,3.836000
qdrant,3,Ingesta repetida + índice listo,"upsert(LOS MISMOS 1500 puntos, batch_size=128) → count() + index_ready()",count() sigue en 1500 y el índice está al día,count() = 1500 · índice: listo,✅ pasa,3.693000
qdrant,4,Búsqueda global,"search(vector[768], top_k=10, filters=[]) · sin filtro","10 resultados, con posición y score tipado",10 resultados · score: similarity (mayor es mejor),✅ pasa,0.022000
qdrant,5,Filtro nativo,"search(vector[768], top_k=10, filters=[brand equals 'einhell']) → el motor filtra por brand_normalized",brand equals 'einhell',"1 resultados, 1 cumplen · auditado contra brand_normalized = einhell",✅ pasa,0.010000
qdrant,6,Lectura por record_id,get('000bd6e8-a995-56d0-ba03-559885ccef39'),devuelve 000bd6e8-a995-56d0-ba03-559885ccef39,encontrado · payload con 8 claves,✅ pasa,0.005000
qdrant,7,Borrado,delete('000bd6e8-a995-56d0-ba03-559885ccef39') → get(mismo id) → count(),000bd6e8-a995-56d0-ba03-559885ccef39 desaparece y count() baja a 1499,"borrado, count() = 1499",✅ pasa,0.009000
qdrant,8,Persistencia tras reinicio,make motor-down MOTOR=… && make motor-up MOTOR=… (+ celda de solo lectura),mismo count() y mismos ids en el top-10 que antes del reinicio,(pendiente),✍️ manual,nan
qdrant,9,Calidad del error con el motor apagado,docker compose -f docker/…/compose.yaml stop → search(),"excepción tipada y legible, no un error genérico",(pendiente),✍️ manual,nan
qdrant,10,Recursos,make motor-stats (con el motor vivo y ya ingerido),RAM del contenedor · tamaño del volumen,(pendiente),✍️ manual,nan


#### ②b Los filtros que el guion no llega a probar

El paso 5 ejerce **una** consulta con **igualdad**, que es el mínimo del enunciado. Esta tabla cubre lo que deja fuera, y las dos cosas son exigencias escritas:

- **Las cuatro consultas de `consultas_filtradas.csv`.** El §5 las pide como evidencia mínima —*"resultados que cumplan la marca en las cuatro consultas"*— y el §8 lo repite como criterio de corrección: *"las consultas filtradas nunca devuelven otra marca"*.
- **El `contains` sobre color.** La sección B lo declaró **requisito duro**, y hasta ahora estaba afirmado desde la documentación de cada SDK en vez de medido.

##### Cómo leer la tabla

| Columna | Qué es |
|---|---|
| `papel` | `obligatorio` = las 4 del enunciado · `referencia`/`palabra`/`fragmento` = las sondas de color |
| **`n_en_catalogo`** | **el oráculo**: cuántos cumplen según pandas, sobre los mismos 1.500, sin motor |
| `n_resultados` | cuántos devolvió el motor, **topado en `top_k`** |
| `cumplen` · `todos_cumplen` | cuántos satisfacen de verdad la condición, auditado contra la clave normalizada |
| **`veredicto`** | confronta las dos columnas anteriores |
| `clave_auditada` | contra qué clave se comprobó (`brand_normalized`, no `brand`) |

##### Por qué hace falta el oráculo

Un `0` del motor **no dice nada por sí solo**: puede que esa marca no tenga productos entre los 1.500 de la muestra o que el filtro esté roto, y las dos cosas se leen igual. Es la ambigüedad que resolvió `cero_por_falta_de_dato` en B.5, y la misma idea que el enunciado aplica a la fidelidad ANN — *"comparar IDs con un oráculo exacto"*.

El oráculo es `filter_reach` sobre **`muestra`**, el corpus ingerido: contra el catálogo completo daría un número que el motor no puede alcanzar.

| Veredicto | Qué significa |
|---|---|
| ✅ ausencia real | El catálogo tampoco tiene ninguno. El cero es correcto |
| ✅ coincide con el catálogo | El motor devolvió lo que había, topado en `top_k` |
| ❌ **FILTRO ROTO** | El catálogo tiene N y el motor devolvió 0 |
| ❌ devuelve de más | El filtro dejó pasar lo que no cumplía |
| ⚠️ legítimo si filtra por palabras | Solo en sondas de `contains`: el oráculo cuenta **subcadenas**, así que un motor de nivel 2 devuelve menos **a propósito** |

**El truco del `fragmento`:** `negr` es subcadena de `negro` pero **no** una palabra suelta. Un motor de subcadena literal lo encuentra; uno que tokeniza, no. Eso clasifica el nivel sin creerse ninguna documentación:

| Nivel | Qué significa |
|---|---|
| **1 · no soportado** | Descarta al motor por el requisito duro |
| **2 · por palabras** | Alcanza los compuestos de B.1 **sin** los falsos positivos de B.3. El mejor |
| **3 · subcadena** | Alcanza lo mismo, y además trae los falsos positivos que B.3 cuantificó |

> ⚠️ Una consulta con **cero resultados no es un fallo del filtro**: puede que esa marca no tenga productos en la muestra de 1.500. Por eso el recuento de aprobadas se hace solo sobre las que devolvieron algo, y las vacías se cuentan aparte.

In [ ]:
probar_filtros("qdrant")

qdrant · marca: 3/3 consultas con resultados, todas de la marca pedida
qdrant · ceros : 1 sin resultados — de ellos 0 son FILTRO ROTO según el catálogo
qdrant · contains: ✅ NIVEL 2 · por palabras — alcanza B.1 SIN los falsos positivos de B.3


caso,papel,filtro,n_en_catalogo,n_resultados,cumplen,todos_cumplen,veredicto,clave_auditada,valores,como_se_lee
FILTER-001,obligatorio,brand equals 'einhell',1,1,1,True,"✅ coincide con el catálogo (1, topado en 10)",brand_normalized,einhell,'herramienta inalámbrica para perforar' — la marca no aparece en el texto
FILTER-002,obligatorio,brand equals 'apple',0,0,0,False,✅ ausencia real — el catálogo tampoco tiene ninguno,brand_normalized,(ninguno),'tableta ligera para estudiar y tomar apuntes' — la marca no aparece en el texto
FILTER-003,obligatorio,brand equals 'nike',6,6,6,True,"✅ coincide con el catálogo (6, topado en 10)",brand_normalized,nike,'zapatillas cómodas para salir a correr' — la marca no aparece en el texto
FILTER-004,obligatorio,brand equals 'samsung',5,5,5,True,"✅ coincide con el catálogo (5, topado en 10)",brand_normalized,samsung,'monitor para trabajar con varias ventanas' — la marca no aparece en el texto
color · igualdad,referencia,color equals 'negro',164,10,10,True,"✅ coincide con el catálogo (164, topado en 10)",color_normalized,negro,referencia: cuántos casan con el valor entero
color · contiene palabra,palabra,color contains 'negro',215,10,10,True,"✅ coincide con el catálogo (215, topado en 10)",color_normalized,"naranja y negro, naranja/negro, negro","si supera a la igualdad, alcanza los compuestos de B.1"
color · contiene fragmento,fragmento,color contains 'negr',218,0,0,False,✅ 0 de 218 — NO encuentra el fragmento ⇒ filtra por palabras (nivel 2),color_normalized,(ninguno),0 ⇒ por palabras (nivel 2) · >0 ⇒ subcadena literal (nivel 3)


#### ③ Paso 10 · recursos — **con el motor vivo**

```bash
make motor-stats
```

Va **antes** que el paso 9: con el motor apagado la medición no vale nada. La salida entera —comando incluido— se pega en `artifacts/recursos/qdrant.txt`, y F.3d deriva la fila 10 de ese fichero para que ningún número se recopie a mano.

#### ④ Paso 8 · persistencia

Ejecuta la celda de abajo, **después** reinicia, y **después** vuelve a ejecutarla con `"despues"`:

```bash
make motor-down MOTOR=qdrant && make motor-up MOTOR=qdrant
```

> ⚠️ **No mires el número absoluto, mira que no cambie.** El paso 7 borró un punto y ②b repuso ese mismo punto para poder comparar contra el oráculo, así que la colección debería tener los **1.500**. Lo que el paso 8 comprueba es que ese recuento —sea cual sea— **sobreviva al reinicio**, y que el top-10 traiga los mismos ids. Un cambio entre «antes» y «después» es el fallo, no el valor en sí.

In [ ]:
instantanea("qdrant", "antes")

qdrant · antes: count=1500 · top-10 leído
  (falta la otra mitad: reinicia el motor y llama con la otra etiqueta)


In [ ]:
instantanea("qdrant", "despues")

qdrant · despues: count=1500 · top-10 leído
  recuento : 1500 → 1500
  conjunto : los mismos ids
  orden    : igual
  score    : se mueve como mucho 0.00e+00

  ✅ idéntico — count() 1500 y los mismos 10 ids en el mismo orden
  → fila 8 de qdrant: ✅ pasa


#### ⑤ Paso 9 · calidad del error — **el último, deja el motor caído**

```bash
docker compose -f docker/qdrant/compose.yaml stop
```

In [ ]:
probar_caido("qdrant")

⚠️ tipada pero de grpc, no del SDK (qdrant_client) — grpc._channel._InactiveRpcError: <_InactiveRpcError of RPC that terminated with: status = StatusCode.UNAVAILABLE details = "failed to connect to all addresses; last error: UNAVAILABLE: ipv4:127
  → fila 9 de qdrant: ✅ pasa


c:\Users\asus\Master\modulos\modulo10_bbdd\practica\AURUM_MARKET\aurum-market-catalog\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:290: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(


#### ⑥ Bajar antes del siguiente

```bash
make motor-down MOTOR=qdrant
```

---

### F.3b · Weaviate

Dos cosas propias de este motor que la tabla debería reflejar:

- **Devuelve distancia, no similitud** — menor es mejor. El paso 4 lo anota en `observado`; si ahí pusiera `similarity`, el adaptador estaría mintiendo y la comparativa ordenaría al revés.
- **La dimensión no se declara:** Weaviate la fija con el primer vector escrito. El paso 1 no puede verificarla, así que la comprobación real es que el paso 4 devuelva algo coherente.

#### ① Levantar — con Qdrant ya abajo

```bash
make motor-down MOTOR=qdrant     # por si acaso
make motor-up   MOTOR=weaviate
```

#### ② Pasos 1–7 · automáticos

In [10]:
probar("weaviate")

NameError: name 'probar' is not defined

#### ②b Los filtros — las 4 consultas y el `contains`

Weaviate resuelve el `contains` con `like "*valor*"`, que es **subcadena literal**. Si la tabla dice nivel 2 en vez de 3, el adaptador está haciendo otra cosa de la que cree.

In [ ]:
probar_filtros("weaviate")

weaviate · marca: 3/3 consultas con resultados, todas de la marca pedida
weaviate · ceros : 1 sin resultados — de ellos 0 son FILTRO ROTO según el catálogo
weaviate · contains: ✅ NIVEL 3 · subcadena literal — alcanza B.1 y trae los falsos positivos de B.3


caso,papel,filtro,n_en_catalogo,n_resultados,cumplen,todos_cumplen,veredicto,clave_auditada,valores,como_se_lee
FILTER-001,obligatorio,brand equals 'einhell',1,1,1,True,"✅ coincide con el catálogo (1, topado en 10)",brand_normalized,einhell,'herramienta inalámbrica para perforar' — la marca no aparece en el texto
FILTER-002,obligatorio,brand equals 'apple',0,0,0,False,✅ ausencia real — el catálogo tampoco tiene ninguno,brand_normalized,(ninguno),'tableta ligera para estudiar y tomar apuntes' — la marca no aparece en el texto
FILTER-003,obligatorio,brand equals 'nike',6,6,6,True,"✅ coincide con el catálogo (6, topado en 10)",brand_normalized,nike,'zapatillas cómodas para salir a correr' — la marca no aparece en el texto
FILTER-004,obligatorio,brand equals 'samsung',5,5,5,True,"✅ coincide con el catálogo (5, topado en 10)",brand_normalized,samsung,'monitor para trabajar con varias ventanas' — la marca no aparece en el texto
color · igualdad,referencia,color equals 'negro',164,10,8,False,"✅ coincide con el catálogo (164, topado en 10)",color_normalized,"naranja y negro, naranja/negro, negro",referencia: cuántos casan con el valor entero
color · contiene palabra,palabra,color contains 'negro',215,10,10,True,"✅ coincide con el catálogo (215, topado en 10)",color_normalized,"naranja y negro, naranja/negro, negro","si supera a la igualdad, alcanza los compuestos de B.1"
color · contiene fragmento,fragmento,color contains 'negr',218,10,10,True,✅ 10 de 218 — encuentra el fragmento ⇒ subcadena literal (nivel 3),color_normalized,"naranja y negro, naranja/negro, negro",0 ⇒ por palabras (nivel 2) · >0 ⇒ subcadena literal (nivel 3)


#### ③ Paso 10 · recursos · ④ Paso 8 · persistencia

```bash
make motor-stats                                     # ③ con el motor vivo
make motor-down MOTOR=weaviate && make motor-up MOTOR=weaviate   # ④
```

La salida de ③, entera, a `artifacts/recursos/weaviate.txt`.

In [ ]:
instantanea("weaviate", "antes")

weaviate · antes: count=1500 · top-10 leído
  (falta la otra mitad: reinicia el motor y llama con la otra etiqueta)


In [ ]:
instantanea("weaviate", "despues")

weaviate · despues: count=1500 · top-10 leído
  recuento : 1500 → 1500
  conjunto : los mismos ids
  orden    : igual
  score    : se mueve como mucho 0.00e+00

  ✅ idéntico — count() 1500 y los mismos 10 ids en el mismo orden
  → fila 8 de weaviate: ✅ pasa


#### ⑤ Paso 9 · calidad del error

```bash
docker compose -f docker/weaviate/compose.yaml stop
```

In [ ]:
probar_caido("weaviate")

✅ tipada por el SDK — weaviate.exceptions.WeaviateConnectionError: Connection to Weaviate failed. Details: Error: timed out. Is Weaviate running and reachable at http://localhost:8080?
  → fila 9 de weaviate: ✅ pasa


#### ⑥ Bajar antes del siguiente

```bash
make motor-down MOTOR=weaviate
```

---

### F.3c · Milvus

> ⚠️ **El último, y con los otros dos abajo.** Son tres contenedores (etcd + minio + standalone) y es el candidato pesado para 7,9 GB.

Dos puntos donde espero que falle o se comporte distinto:

- **`pymilvus` instalado es 3.x** y el adaptador se escribió contra la API 2.x documentada. Es el sospechoso principal de la primera pasada.
- **El `like "%valor%"` con comodín por delante.** De eso depende que cumpla el requisito duro: las versiones antiguas de Milvus solo resolvían prefijos. Aquí solo se usa `equals`, así que **el paso 5 pasando no demuestra que el `contains` funcione** — eso hay que probarlo aparte antes de darlo por bueno para el color.

#### ① Levantar — con los otros dos abajo

```bash
make all-down                    # apaga los tres, por si queda alguno
make motor-up MOTOR=milvus
```

#### ② Pasos 1–7 · automáticos

In [8]:
probar("milvus")

milvus · versión del servidor: pymilvus 3.0.1
anotaciones manuales conservadas: [8, 9, 10] — el paso 8 hay que rehacerlo, la coleccion es otra


motor,paso,comprobacion,que_ha_hecho,esperado,observado,resultado,segundos
milvus,1,Crear colección,"create_collection(dim=768, metric='cosine', recreate=True)","dim=768, métrica=cosine","creada (dim=768, métrica=cosine)",✅ pasa,3.283000
milvus,2,Ingesta por lotes,"upsert(1500 puntos, batch_size=128) → count()",count() == 1500,count() = 1500,✅ pasa,4.635000
milvus,3,Ingesta repetida + índice listo,"upsert(LOS MISMOS 1500 puntos, batch_size=128) → count() + index_ready()",count() sigue en 1500 y el índice está al día,count() = 1500 · índice: no lo reporta el motor,✅ pasa,10.188000
milvus,4,Búsqueda global,"search(vector[768], top_k=10, filters=[]) · sin filtro","10 resultados, con posición y score tipado",10 resultados · score: similarity (mayor es mejor),✅ pasa,0.934000
milvus,5,Filtro nativo,"search(vector[768], top_k=10, filters=[brand equals 'einhell']) → el motor filtra por brand_normalized",brand equals 'einhell',"1 resultados, 1 cumplen · auditado contra brand_normalized = einhell",✅ pasa,0.028000
milvus,6,Lectura por record_id,get('000bd6e8-a995-56d0-ba03-559885ccef39'),devuelve 000bd6e8-a995-56d0-ba03-559885ccef39,encontrado · payload con 8 claves,✅ pasa,0.009000
milvus,7,Borrado,delete('000bd6e8-a995-56d0-ba03-559885ccef39') → get(mismo id) → count(),000bd6e8-a995-56d0-ba03-559885ccef39 desaparece y count() baja a 1499,"borrado, count() = 1499",✅ pasa,8.549000
milvus,8,Persistencia tras reinicio,make motor-down MOTOR=… && make motor-up MOTOR=… (+ celda de solo lectura),mismo count() y mismos ids en el top-10 que antes del reinicio,✅ idéntico — count() 1500 y los mismos 10 ids en el mismo orden,✅ pasa,nan
milvus,9,Calidad del error con el motor apagado,docker compose -f docker/…/compose.yaml stop → search(),"excepción tipada y legible, no un error genérico",✅ tipada por el SDK — pymilvus.exceptions.MilvusException:,✅ pasa,nan
milvus,10,Recursos,make motor-stats (con el motor vivo y ya ingerido),RAM del contenedor · tamaño del volumen,"RAM 530.8 MiB en 3 contenedores, el mayor aurum-market-milvus-minio · volumen 182.9 MB (no comparable entre motores) · dentro del `mem_limit` declarado, el más apretado aurum-market-milvus-minio al 50% · foto con el contenedor «Up 2 minutes (healthy)»",✅ pasa,nan


#### ②b Los filtros — **aquí se juega el requisito duro**

Esta es la celda que decide si Milvus sirve. El adaptador usa `like "%valor%"` con **comodín por delante**, y las versiones antiguas de Milvus solo resolvían prefijos: si esa sonda devuelve `NO SOPORTADO` o cero, Milvus queda descartado por el requisito duro de la sección B, sin importar cómo le haya ido en los pasos 1–7.

In [9]:
probar_filtros("milvus")

milvus · marca: 3/3 consultas con resultados, todas de la marca pedida
milvus · ceros : 1 sin resultados — de ellos 0 son FILTRO ROTO según el catálogo
milvus · contains: ✅ NIVEL 3 · subcadena literal — alcanza B.1 y trae los falsos positivos de B.3


caso,papel,filtro,n_en_catalogo,n_resultados,cumplen,todos_cumplen,veredicto,clave_auditada,valores,como_se_lee
FILTER-001,obligatorio,brand equals 'einhell',1,1,1,True,"✅ coincide con el catálogo (1, topado en 10)",brand_normalized,einhell,'herramienta inalámbrica para perforar' — la marca no aparece en el texto
FILTER-002,obligatorio,brand equals 'apple',0,0,0,False,✅ ausencia real — el catálogo tampoco tiene ninguno,brand_normalized,(ninguno),'tableta ligera para estudiar y tomar apuntes' — la marca no aparece en el texto
FILTER-003,obligatorio,brand equals 'nike',6,6,6,True,"✅ coincide con el catálogo (6, topado en 10)",brand_normalized,nike,'zapatillas cómodas para salir a correr' — la marca no aparece en el texto
FILTER-004,obligatorio,brand equals 'samsung',5,5,5,True,"✅ coincide con el catálogo (5, topado en 10)",brand_normalized,samsung,'monitor para trabajar con varias ventanas' — la marca no aparece en el texto
color · igualdad,referencia,color equals 'negro',164,10,10,True,"✅ coincide con el catálogo (164, topado en 10)",color_normalized,negro,referencia: cuántos casan con el valor entero
color · contiene palabra,palabra,color contains 'negro',215,10,10,True,"✅ coincide con el catálogo (215, topado en 10)",color_normalized,"naranja y negro, naranja/negro, negro","si supera a la igualdad, alcanza los compuestos de B.1"
color · contiene fragmento,fragmento,color contains 'negr',218,10,10,True,✅ 10 de 218 — encuentra el fragmento ⇒ subcadena literal (nivel 3),color_normalized,"naranja y negro, naranja/negro, negro",0 ⇒ por palabras (nivel 2) · >0 ⇒ subcadena literal (nivel 3)


#### ③ Paso 10 · recursos — **suma los tres contenedores**

```bash
make motor-stats
```

La salida entera va a `artifacts/recursos/milvus.txt`, y F.3d la suma sola.

> ⚠️ Tienen que salir **etcd + minio + standalone**, no solo `standalone`: es lo que de verdad cuesta tener Milvus en marcha, y es el criterio por el que el plan lo marca como pesado.
>
> **Attu queda fuera.** Es una herramienta de inspección, no parte del motor: sumarla haría que Milvus pareciera más caro frente a Qdrant, que sirve su panel desde el mismo proceso. Levántala después de anotar, con `docker compose -f docker/milvus/compose.yaml up -d attu` (<http://localhost:8000>).

#### ④ Paso 8 · persistencia

```bash
make motor-down MOTOR=milvus && make motor-up MOTOR=milvus
```

In [10]:
instantanea("milvus", "antes")

milvus · antes: count=1500 · top-10 leído
  (falta la otra mitad: reinicia el motor y llama con la otra etiqueta)


In [ ]:
instantanea("milvus", "despues")

milvus · despues: count=1500 · top-10 leído
  recuento : 1500 → 1500
  conjunto : los mismos ids
  orden    : igual
  score    : se mueve como mucho 0.00e+00

  ✅ idéntico — count() 1500 y los mismos 10 ids en el mismo orden
  → fila 8 de milvus: ✅ pasa


#### ⑤ Paso 9 · calidad del error

```bash
docker compose -f docker/milvus/compose.yaml stop
```

In [ ]:
probar_caido("milvus")

2026-08-22 21:10:55,817 [WARNING][_recover]: Connection recovery failed (connection_manager.py:676)
Traceback (most recent call last):
  File "c:\Users\asus\Master\modulos\modulo10_bbdd\practica\AURUM_MARKET\aurum-market-catalog\.venv\Lib\site-packages\pymilvus\client\grpc_handler.py", line 257, in _wait_for_channel_ready
    target_final_channel, target_stub = self._setup_identifier_interceptor(
                                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\asus\Master\modulos\modulo10_bbdd\practica\AURUM_MARKET\aurum-market-catalog\.venv\Lib\site-packages\pymilvus\client\grpc_handler.py", line 450, in _setup_identifier_interceptor
    else self._internal_register(user, host, stub=target_stub, timeout=timeout)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\asus\Master\modulos\modulo10_bbdd\practica\AURUM_MARKET\aurum-market-catalog\.venv\Lib\site-packages\pymilvus\client\grpc_handler.py", line 3267, in _in

✅ tipada por el SDK — pymilvus.exceptions.MilvusException: <MilvusException: (code=2, message=Fail connecting to server on localhost:19530, illegal connection params or server unavailable)>
  → fila 9 de milvus: ✅ pasa


#### ⑥ Apagar y devolver el permiso

```bash
make all-down
```

Y pon `AURUM_ALLOW_RESET=false` en el `.env`: el permiso de borrar solo hacía falta mientras durase esta sección.

---

## F.3d · El paso 10, desde los transcritos de la terminal

`docker stats` y `docker system df` se ejecutan en la terminal, así que lo que entra al repo es el texto pegado tal cual en `artifacts/recursos/{motor}.txt`. Esta celda lo parsea: la tabla se **deriva** del transcrito en vez de recopiarse a mano, que es donde se cuela un número que ya no corresponde a lo que se midió.

### Lo que la tabla dice, y lo que no

| Columna | Cómo se lee |
|---|---|
| `ram_mib` | Suma de los contenedores del motor. Milvus son tres; `attu` queda fuera por la regla escrita en su `compose.yaml` |
| `dentro_del_limite` | El **único criterio del paso 10 que estaba declarado antes de medir**: el `mem_limit` de cada compose. Inventar ahora un umbral en MiB sería ponerle la vara al ganador |
| `mas_apretado` | Qué contenedor va más justo dentro de su propio límite |
| `volumen_mb` | ⚠️ **No comparable entre motores** — ver abajo |
| `en_marcha` | Cuánto llevaba vivo el contenedor al tomar la foto |

**Por qué el volumen no compara.** Los volúmenes con nombre sobreviven a `down`, así que cada uno arrastra todo lo que ese motor escribió mientras se desarrollaba su adaptador, más la preasignación de WAL —los 144,9 MB de `etcd` son fichero reservado, no catálogo—. Los mismos 1.500 × 768 son 4,6 MB en los tres: esa columna mide edad del volumen, no eficiencia. El número que sí valdrá es el de la sección G, con colección nueva y un solo motor.

**Y la RAM es una foto en reposo**, no un pico bajo carga. Con 4,6 MB de dato dentro, lo que se está midiendo es el coste base del proceso — que es justamente lo que separa a un contenedor de tres.

In [14]:
# 📄 DATOS · artifacts/recursos/{motor}.txt — lo que imprimió `make motor-stats`
RECURSOS = Path("..") / "artifacts" / "recursos"
# Attu es panel de inspección, no motor: sumarlo haría a Milvus más caro
# frente a Qdrant, que sirve el suyo desde el mismo proceso.
recursos = resource_table(RECURSOS, CANDIDATOS, exclude=("aurum-market-milvus-attu",))

for _, fila in recursos.iterrows():
    if fila["motor"] in HUMO:
        anotar(fila["motor"], 10, resource_note(fila), bool(fila["dentro_del_limite"]))

sin_medir = [m for m in CANDIDATOS if m not in set(recursos["motor"])]
print(f"sin transcrito: {sin_medir or 'ninguno'}")
recursos

  → fila 10 de qdrant: ✅ pasa
  → fila 10 de weaviate: ✅ pasa
  → fila 10 de milvus: ✅ pasa
sin transcrito: ninguno


,motor,n_contenedores,ram_mib,volumen_mb,mayor_consumidor,dentro_del_limite,mas_apretado,volumenes,en_marcha,excluidos
0,qdrant,1,36.4,293.8,aurum-market-qdrant,True,aurum-market-qdrant al 2%,aurum-market-qdrant-data,Up 6 minutes (healthy),
1,weaviate,1,50.9,10.9,aurum-market-weaviate,True,aurum-market-weaviate al 2%,aurum-market-weaviate-data,Up 10 minutes (healthy),
2,milvus,3,530.8,182.9,aurum-market-milvus-minio,True,aurum-market-milvus-minio al 50%,"aurum-market-milvus-data, aurum-market-milvus-...",Up 2 minutes (healthy),


---

## F.3e · La anomalía de Milvus, aislada

La primera vez que se midió el paso 8, Milvus mantuvo el recuento pero **devolvió otro top-10**; al repetirlo sobre la misma colección, horas después y sin reingerir, salió idéntico. La anomalía existió y no se reprodujo, y hay que explicar las dos cosas.

### El problema del experimento original

Aquel par de lecturas mezclaba **dos variables**: entre «antes» y «después» pasó el reinicio, pero también pasó el tiempo — y la primera lectura se tomó justo tras ingerir 1.500 puntos y el `upsert` de reposición de ②b. Con dos lecturas no se sabe cuál causó el cambio.

La hipótesis es el estado interno de segmentos —Milvus sirve desde segmentos en crecimiento y desde otros sellados e indexados, y una consulta puede ver un reparto distinto según cuándo llegue—, pero es **hipótesis, no causa medida**.

### Tres lecturas en vez de dos

| Lectura | Cuándo |
|---|---|
| `recien_ingerido` | inmediatamente después de `probar("milvus")`, sin esperar |
| `asentado` | dos o tres minutos después, **sin reiniciar nada** |
| `tras_reinicio` | después de `down` + `up` |

Cada par aísla una cosa:

| Par | Qué aísla | Si cambia ahí |
|---|---|---|
| `recien_ingerido` → `asentado` | **el tiempo**, sin reinicio de por medio | El índice se estaba asentando. El reordenamiento no lo causa el reinicio, sino consultar antes de que el índice esté al día |
| `asentado` → `tras_reinicio` | **el reinicio**, con el índice ya asentado | Es del reinicio: Milvus recarga o reconstruye de otra forma. Este par **es** el paso 8 |
| `recien_ingerido` → `tras_reinicio` | las dos a la vez | Es lo que se midió la primera vez, y por eso no distinguía |

> ⚡ **Por qué esto importa más allá de Milvus.** Si el cambio aparece en el primer par, conecta directamente con el paso 3: Milvus es de los que responden `índice: no lo reporta el motor`, y §3.2 pide *"verificad el recuento final y el estado de indexación antes de aceptar consultas"*. Un motor que no informa de cuándo su índice está al día, y que mientras tanto devuelve otro top-10 para la misma consulta, es exactamente el riesgo que ese requisito cubre. Qdrant lo reporta y no divergió ninguna de las dos veces.

Si no cambia en ningún par, la conclusión también vale: la anomalía no es reproducible y se declara como observación única, sin atribuirle una causa que no se ha medido.

#### ① Reingerir en limpio

```bash
make motor-up MOTOR=milvus
```

> 🔒 Recrea la colección, así que necesita `AURUM_ALLOW_RESET=true` en el `.env`.
>
> Las anotaciones manuales de Milvus **se conservan** al reejecutar `probar`, pero la del paso 8 quedará pendiente de rehacer: la colección es otra.

In [11]:
probar("milvus")

milvus · versión del servidor: pymilvus 3.0.1
anotaciones manuales conservadas: [8, 9, 10] — el paso 8 hay que rehacerlo, la coleccion es otra


motor,paso,comprobacion,que_ha_hecho,esperado,observado,resultado,segundos
milvus,1,Crear colección,"create_collection(dim=768, metric='cosine', recreate=True)","dim=768, métrica=cosine","creada (dim=768, métrica=cosine)",✅ pasa,3.664000
milvus,2,Ingesta por lotes,"upsert(1500 puntos, batch_size=128) → count()",count() == 1500,count() = 1500,✅ pasa,2.641000
milvus,3,Ingesta repetida + índice listo,"upsert(LOS MISMOS 1500 puntos, batch_size=128) → count() + index_ready()",count() sigue en 1500 y el índice está al día,count() = 1500 · índice: no lo reporta el motor,✅ pasa,9.683000
milvus,4,Búsqueda global,"search(vector[768], top_k=10, filters=[]) · sin filtro","10 resultados, con posición y score tipado",10 resultados · score: similarity (mayor es mejor),✅ pasa,0.036000
milvus,5,Filtro nativo,"search(vector[768], top_k=10, filters=[brand equals 'einhell']) → el motor filtra por brand_normalized",brand equals 'einhell',"1 resultados, 1 cumplen · auditado contra brand_normalized = einhell",✅ pasa,0.088000
milvus,6,Lectura por record_id,get('000bd6e8-a995-56d0-ba03-559885ccef39'),devuelve 000bd6e8-a995-56d0-ba03-559885ccef39,encontrado · payload con 8 claves,✅ pasa,0.014000
milvus,7,Borrado,delete('000bd6e8-a995-56d0-ba03-559885ccef39') → get(mismo id) → count(),000bd6e8-a995-56d0-ba03-559885ccef39 desaparece y count() baja a 1499,"borrado, count() = 1499",✅ pasa,11.551000
milvus,8,Persistencia tras reinicio,make motor-down MOTOR=… && make motor-up MOTOR=… (+ celda de solo lectura),mismo count() y mismos ids en el top-10 que antes del reinicio,✅ idéntico — count() 1500 y los mismos 10 ids en el mismo orden,✅ pasa,nan
milvus,9,Calidad del error con el motor apagado,docker compose -f docker/…/compose.yaml stop → search(),"excepción tipada y legible, no un error genérico",✅ tipada por el SDK — pymilvus.exceptions.MilvusException:,✅ pasa,nan
milvus,10,Recursos,make motor-stats (con el motor vivo y ya ingerido),RAM del contenedor · tamaño del volumen,"RAM 530.8 MiB en 3 contenedores, el mayor aurum-market-milvus-minio · volumen 182.9 MB (no comparable entre motores) · dentro del `mem_limit` declarado, el más apretado aurum-market-milvus-minio al 50% · foto con el contenedor «Up 2 minutes (healthy)»",✅ pasa,nan


#### ② La lectura inmediata — **sin esperar, es la mitad del experimento**

In [12]:
instantanea("milvus", "recien_ingerido")

milvus · recien_ingerido: count=1499 · top-10 leído


#### ③ La misma lectura, dos o tres minutos después y **sin tocar el motor**

In [13]:
instantanea("milvus", "asentado")

milvus · asentado: count=1499 · top-10 leído


#### ④ Ahora sí, el reinicio

```bash
make motor-down MOTOR=milvus && make motor-up MOTOR=milvus
```

In [14]:
instantanea("milvus", "tras_reinicio")

milvus · tras_reinicio: count=1499 · top-10 leído


#### ⑤ Los tres pares, y la fila 8 definitiva

La fila 8 se anota con el par `asentado → tras_reinicio`, que es el que responde a lo que el paso 8 pregunta: si el estado sobrevive a un reinicio. El otro par es diagnóstico y no entra en la tabla — explica *por qué*, no *si*.

In [15]:
tiempo = comparar("milvus", "recien_ingerido", "asentado")
reinicio = comparar("milvus", "asentado", "tras_reinicio")
mezclado = comparar("milvus", "recien_ingerido", "tras_reinicio")

print()
if not tiempo.same_order:
    print("→ el índice se estaba asentando: el reinicio no era la causa.")
    print("  Conecta con el paso 3: Milvus no reporta el estado de indexación.")
elif not reinicio.same_order:
    print("→ es el reinicio, con el índice ya asentado.")
else:
    print("→ no se reproduce por ninguna vía: queda como observación única.")

# El par que ES el paso 8. Sustituye a la anotación anterior de Milvus.
anotar("milvus", 8, reinicio.verdict(), reinicio.passed)

milvus · recien_ingerido → asentado
  ✅ idéntico — count() 1499 y los mismos 10 ids en el mismo orden
milvus · asentado → tras_reinicio
  ✅ idéntico — count() 1499 y los mismos 10 ids en el mismo orden
milvus · recien_ingerido → tras_reinicio
  ✅ idéntico — count() 1499 y los mismos 10 ids en el mismo orden

→ no se reproduce por ninguna vía: queda como observación única.
  → fila 8 de milvus: ✅ pasa


```bash
make motor-down MOTOR=milvus
```

---

## F.4 · La comparativa, y el artefacto

Cuando los tres motores hayan pasado por F.3, esta celda junta las tres tablas y escribe `artifacts/comparativa_motores.md`, el artefacto de **R03**.

### La tabla de ✅/❌ no elige motor

Si los tres pasan los diez pasos —y los pasan—, esa tabla demuestra que los tres **sirven**, que era la pregunta eliminatoria, pero no separa a ninguno. Lo que separa está en la columna `observado`, y por eso el artefacto lleva además:

| Sección | Qué aporta a R03 |
|---|---|
| **Dónde no se comportaron igual** | Solo los pasos con `observado` distinto entre motores. Las filas donde coinciden no deciden nada y estorban |
| **Segundos por paso** | El coste de cada operación. Ojo: son de una sola pasada sobre 1.500 puntos, no un banco de pruebas |
| **Nivel de `contains`** | El requisito duro de la sección B |
| **Paso 10 · recursos** | Con sus condiciones de medición declaradas |

> **R03 no se elige aquí, se lee.** El motor sale de esta tabla contrastada con los criterios de la sección A y con los seis de la sesión 1 —memoria del índice y dependencia del proveedor son los que más se mueven al elegir motor—. Escribir la conclusión antes de tener las tres filas sería elegir primero y justificar después.

In [17]:
# Requiere haber ejecutado F.3 con cada motor. Con uno solo, la tabla se
# genera igual y deja claro cuáles faltan: un artefacto a medias que dice
# que lo está es más útil que ninguno.
# No hace falta tener el kernel de F.3 vivo: `HUMO` se rellena desde
# artifacts/humo/ si la memoria está vacía (§8, artefactos regenerables).
faltan = [m for m in CANDIDATOS if m not in HUMO]
print(f"motores medidos: {list(HUMO) or 'ninguno todavía'}")
print(f"faltan         : {faltan or 'ninguno'}")

if HUMO:
    comparativa = pd.concat(
        [smoke_table(pasos, motor=nombre) for nombre, pasos in HUMO.items()]
    )
    resumen = comparativa.pivot_table(
        index=["paso", "comprobacion"], columns="motor",
        values="resultado", aggfunc="first",
    )
    display(resumen)

    destino = Path("..") / "artifacts" / "comparativa_motores.md"
    cabecera = [
        "# Prueba de humo · comparativa de motores (D12 → R03)",
        "",
        f"Corpus: `catalogo_muestra.csv` ({len(puntos)} puntos) · "
        f"dim {DIM} · métrica cosine · lote {LOTE} (D15)",
        f"Payload: `{ESQUEMA}` (D13) · nulos: `{POLITICA_NULOS}` (D14)",
        f"Filtro del paso 5: `{FILTRO[0].field} {FILTRO[0].operator} "
        f"{FILTRO[0].value!r}` ({consulta['workload_id']})",
        "",
    ]
    if faltan:
        cabecera += [f"> ⚠️ Tabla incompleta: falta medir {', '.join(faltan)}.", ""]

    # El nivel de `contains` de cada motor: es el requisito duro de la
    # sección B y no sale de la tabla de pasos, así que se añade aparte.
    niveles = [
        "", "## Nivel de `contains` sobre metadatos (requisito duro · sección B)", "",
        *(f"- **{m}** — {contains_level(t)}" for m, t in FILTROS.items()),
        "",
        "## Las cuatro consultas filtradas (§5)", "",
    ]
    if FILTROS:
        obligatorias = pd.concat([
            t[t["papel"] == "obligatorio"].assign(motor=m) for m, t in FILTROS.items()
        ])
        niveles.append(
            obligatorias.pivot_table(
                index=["caso", "filtro"], columns="motor",
                values="todos_cumplen", aggfunc="first",
            ).to_markdown()
        )

    # Lo que la tabla de ✅/❌ esconde. Con diez aprobados en los tres, estas
    # son las unicas filas que pueden decidir R03.
    diferencias = smoke_differences(HUMO)
    if not diferencias.empty:
        niveles += [
            "", "## Donde los motores NO se comportaron igual", "",
            "Solo los pasos con `observado` distinto entre motores: los que "
            "coinciden no separan a nadie.", "",
            diferencias.to_markdown(), "",
        ]
    tiempos = comparativa.pivot_table(
        index=["paso", "comprobacion"], columns="motor",
        values="segundos", aggfunc="first",
    ).dropna(how="all")
    if not tiempos.empty:
        niveles += [
            "", "## Segundos por paso", "",
            "Una sola pasada sobre 1.500 puntos, no un banco de pruebas: "
            "sirven para ver ordenes de magnitud, no para afinar.", "",
            tiempos.round(3).to_markdown(), "",
        ]

    # El paso 10 y las condiciones en que se midió. Van al artefacto y no
    # solo a la celda: quien lea la comparativa tiene que poder ver que el
    # volumen no compara ANTES de usarlo para elegir motor.
    recursos = resource_table(
        Path("..") / "artifacts" / "recursos", CANDIDATOS,
        exclude=("aurum-market-milvus-attu",),
    )
    if not recursos.empty:
        niveles += [
            "", "## Paso 10 · recursos, y en qué condiciones se midieron", "",
            recursos.to_markdown(index=False), "",
            *(f"- {aviso}" for aviso in MEDICION_ADVERTENCIAS),
            "",
            "Transcritos íntegros de `docker stats` y `docker system df -v` en "
            "`artifacts/recursos/`; la tabla de arriba se deriva de ellos.",
        ]

    destino.write_text(
        "\n".join(cabecera) + resumen.to_markdown() + "\n"
        + "\n".join(niveles) + "\n",
        encoding="utf-8",
    )
    print(f"\nEscrito {destino}")

motores medidos: ['milvus', 'qdrant', 'weaviate']
faltan         : ninguno


,motor,milvus,qdrant,weaviate
paso,comprobacion,,,
1,Crear colección,✅ pasa,✅ pasa,✅ pasa
2,Ingesta por lotes,✅ pasa,✅ pasa,✅ pasa
3,Ingesta repetida + índice listo,✅ pasa,✅ pasa,✅ pasa
4,Búsqueda global,✅ pasa,✅ pasa,✅ pasa
5,Filtro nativo,✅ pasa,✅ pasa,✅ pasa
6,Lectura por record_id,✅ pasa,✅ pasa,✅ pasa
7,Borrado,✅ pasa,✅ pasa,✅ pasa
8,Persistencia tras reinicio,✅ pasa,✅ pasa,✅ pasa
9,Calidad del error con el motor apagado,✅ pasa,✅ pasa,✅ pasa



Escrito ..\artifacts\comparativa_motores.md


---

# G · El índice definitivo: los 15.000 sobre Qdrant

R03 eligió Qdrant. Aquí se construye la colección que van a usar NB05, NB06 y NB08, y se comprueba que **es la que se cree que es**, que no es lo mismo que comprobar que se creó.

### Qué cambia respecto a la prueba de humo

| | F · humo | G · índice |
|---|---|---|
| Corpus | 🔬 1.500 desechables | 📚 **15.000, los de verdad** |
| Colección | `aurum_humo_*`, se recrea en cada pasada | `aurum_catalogo__*`, **se conserva** |
| La pregunta | ¿sirve este motor? | ¿es este índice el que creo? |
| `recreate` | `True` por defecto | **`False` por defecto** |
| Pasos manuales | 8, 9 y 10 | **7 y 8** — el error con el motor caído no se repite: es del SDK, no del índice |

### Los dos prefijos no son cosmética

El guion de humo **borra y recrea** su colección cada vez que se ejecuta. Si el índice bueno viviera bajo el mismo prefijo, una errata en un nombre bastaría para llevarse por delante los 15.000 puntos. Con `aurum_catalogo` y `aurum_humo` separados en el guardián, el guion de humo no puede alcanzarlo **ni equivocándose**.

### El nombre lleva el contrato dentro

`aurum_catalogo__gemini_embedding_2__A4__768` — modelo, plantilla y dimensión. Los tres invalidan los vectores guardados si cambian, así que el nombre los declara en vez de dejarlos en la memoria de quien lo creó.

Es además la mitad del control de versionado al que `config.yaml` se comprometió en el criterio 6: **migrar es construir la colección nueva al lado y cambiar el puntero**, nunca reindexar sobre la viva. Con el contrato en el nombre, las dos pueden convivir mientras dure la migración.

### El índice se construye con la configuración ANN por defecto, a propósito

Lo que se prueba en G es **el motor contra el catálogo completo**: que ingiere los 15.000 sin duplicarlos, que el esquema es el declarado, que sobrevive a un reinicio y que los canarios vuelven donde deben. Para eso los parámetros del ANN no hacen falta, y tocarlos aquí sería peor: **D16 —el recall mínimo y el p95 máximo— se fija antes de ver ninguna curva**, y unos valores elegidos ahora se acabarían escribiendo a la medida de lo que saliera.

El estudio del índice —familia, parámetros, fidelidad y latencia— es **NB06**. Allí se barre `ef`, que se ajusta por consulta y no obliga a reconstruir; si llegara a tocar `m` o `ef_construct`, que se fijan al construir, haría falta un índice nuevo al lado — la operación que el criterio 6 ya contempla.

### Antes de empezar: el volumen limpio

El volumen de Qdrant arrastra 293,8 MB de historia del desarrollo del adaptador. Para que el tamaño medido en el paso 8 sea el **del índice** y no el de esa historia, hay que borrarlo antes de construir:

```bash
docker compose -f docker/qdrant/compose.yaml down -v   # -v borra el volumen
make motor-up MOTOR=qdrant
```

Se lleva también la colección de humo, y no pasa nada: sus resultados están en `artifacts/humo/*.json` y la comparativa se regenera sin motor.

## G.1 · Los 15.000 puntos

Mismo montaje que F.1 con dos diferencias: el corpus es el completo y el recorte de A4 se calcula sobre él, así que el corte es la mediana de los 15.000 —**936 caracteres**— y no la de la muestra.

> 💸 **El freno de la celda.** Los vectores tienen que salir de la caché de NB03. Si la clave no casara —una plantilla distinta, otro contrato—, `encode_corpus` se pondría a codificar 15.000 documentos contra la API de pago sin preguntar. La celda comprueba que el `.npy` existe **antes** de llamar, y para si no está.

In [8]:
# 📄 DATOS · 📚 catalogo_productos.csv (15.000) + sus vectores A4 de la caché
from aurum.embeddings import cache_key, corpus_fingerprint
from aurum.motores import (
    ACCEPTANCE_MANUAL_NUMBERS,
    CATALOG_PREFIX,
    catalog_collection_name,
    run_acceptance,
    self_retrieval_canaries,
)

COLECCION = catalog_collection_name(model=MODELO, template=PLANTILLA, dim=DIM)
CORPUS_ID = f"catalogo_productos__{PLANTILLA}"

# A4 sobre el catalogo completo: el corte es la mediana de los 15.000.
textos_completo = render_template(completo, PLANTILLA)

# El freno: mirar la cache ANTES de llamar al codificador. `encode_corpus`
# codificaria sin preguntar, y son 15.000 documentos contra una API de pago.
clave = cache_key(
    model_id=MODELO, kind="document", contract=CONTRATO,
    corpus_id=CORPUS_ID, fingerprint=corpus_fingerprint(textos_completo),
)
if not (CACHE / f"{clave}.npy").exists():
    raise RuntimeError(
        f"Los vectores de {CORPUS_ID} no estan en cache ({clave}).\n"
        f"Codificarlos son 15.000 llamadas de pago, asi que la celda para "
        f"aqui en vez de pagarlas sin avisar. Comprueba que MODELO, "
        f"CONTRATO y PLANTILLA son los de R01/R02 antes de forzar nada."
    )

codificado_completo = encode_corpus(
    GeminiEncoder(api_key=os.environ.get("GEMINI_API_KEY"), model_id=MODELO,
                  native_dim=3072, window=8192),
    textos_completo, corpus_id=CORPUS_ID,
    kind="document", contract=CONTRATO, batch_size=32, cache_dir=CACHE,
)
vectores_completo = truncate_dim(codificado_completo.vectors, DIM)

print(f"coleccion : {COLECCION}")
print(f"vectores  : {vectores_completo.shape} · desde cache: "
      f"{codificado_completo.stats.desde_cache}")
# Normas ~1 y sin NaN/inf: dos de las comprobaciones que pide el plan, y se
# hacen sobre la matriz ANTES de subirla. Un NaN dentro del motor ya no se
# distingue de un vector legitimo.
print(f"salud     : {vector_health(vectores_completo)}")

coleccion : aurum_catalogo__gemini_embedding_2__A4__768
vectores  : (15000, 768) · desde cache: True
salud     : {'n_vectores': 15000, 'dim': 768, 'dtype': 'float32', 'finito': True, 'norma_min': 1.0, 'norma_max': 1.0, 'normalizado': True, 'n_filas_duplicadas': 11, 'bytes_por_vector': 3072}


In [9]:
# 📄 DATOS · 📚 catalogo_productos.csv (15.000)
# Las claves derivadas de D03 y el payload de D13/D14, igual que en F.1.
completo_con_claves = completo.copy()
for campo in CAMPOS_FILTRABLES:
    completo_con_claves = add_normalized_key(
        completo_con_claves, field=campo, mode=NORMALIZACION
    )

puntos_completo = [
    Point(
        record_id=fila["record_id"],
        vector=vectores_completo[i],
        payload=build_payload(
            fila, fields=PAYLOAD_SCHEMAS[ESQUEMA], null_policy=POLITICA_NULOS
        ),
    )
    for i, fila in enumerate(completo_con_claves.to_dict("records"))
]
canarios = self_retrieval_canaries(puntos_completo, n=3)

print(f"{len(puntos_completo)} puntos · "
      f"{len(set(p.record_id for p in puntos_completo))} ids distintos")
print(f"canarios: {[c.record_id for c in canarios]}")

15000 puntos · 15000 ids distintos
canarios: ['e1a0e559-6a49-5be5-b617-ec8a4899e975', 'e89bbc67-b896-5fd3-9d1d-0049f561a914', '7cac59bd-1118-5b88-b2f8-504cec61b4e3']


## G.2 · Construir y aceptar

Seis comprobaciones automáticas. Las cuatro primeras son de contrato y las dos últimas son las que de verdad cuestan encontrar de otra forma:

| Paso | Qué responde |
|---|---|
| 1 · colección | Dimensión y métrica explícitas, con el prefijo del catálogo |
| 2 · ingesta | `count() == 15.000`, y **cuántos vectores por segundo** — el número que va al README |
| 3 · índice al día | §3.2 lo pide **antes** de aceptar consultas. Dos datos: si estaba listo al terminar la ingesta y, si no, **cuánto tardó** |
| 4 · dimensión real | Preguntándosela a la colección, no repitiendo la variable del notebook |
| 5 · idempotencia | Reingerir los mismos 15.000 no puede sumar ni uno |
| **6 · canarios** | Tres puntos buscados **con su propio vector** deben volver los primeros |

### Por qué el canario es la búsqueda de sí mismo

Lo tentador sería usar una consulta de desarrollo con su producto relevante, pero sería un mal canario: que un relevante entre en el top-10 es una pregunta de **calidad**, y con un Recall@10 de 0,26 fallaría a menudo con el índice perfectamente sano. Un canario que salta por lo que no vigila no vigila nada.

Buscar un punto con su propio vector tiene una respuesta que no depende del modelo: **debe volver él, el primero**. Y detecta la avería que ningún recuento ve —vectores y payloads desalineados durante la ingesta por lotes—, que es el fallo silencioso clásico: con recuento, dimensión e idempotencia correctos, un índice desalineado pasa todo lo demás.

### El paso 3 mide dos cosas, y la segunda solo se puede medir aquí

Con 1.500 puntos el índice se construía en un suspiro y el paso salía `listo` siempre. Con 15.000 no: el `count()` ya dice 15.000 mientras el grafo HNSW sigue construyéndose por detrás. En Qdrant eso **no da resultados incorrectos** —busca en exacto sobre los segmentos sin indexar, y por eso los canarios pasan igual—, pero es el momento en que no hay que aceptar tráfico ni medir latencia.

Por eso la fila anota las dos mitades: **si estaba listo al terminar la ingesta** y **cuántos segundos tardó en estarlo**. Sin ese segundo número la alternativa es dormir un rato a ojo, y el enunciado pide lo contrario —*saber esperar, fallar o informar*—: la espera sondea hasta un tope y se rinde con un mensaje claro en vez de colgarse.

> ⏱️ La ingesta de 15.000 ronda los 40 s a ~375 vectores/s. La celda la hace **dos veces** —la segunda es el paso 5— y además puede quedarse esperando al índice, así que cuenta un par de minutos en total.

In [10]:
# 📄 DATOS · 📚 los 15.000 puntos de G.1 · ⚠️ requiere `make motor-up MOTOR=qdrant`
from aurum.motores.qdrant import QdrantStore

indice = QdrantStore(
    collection=COLECCION,
    url=os.environ.get("AURUM_QDRANT_URL", "http://localhost:6333"),
    api_key=os.environ.get("AURUM_QDRANT_API_KEY"),
    prefix=CATALOG_PREFIX,   # el guardian del indice bueno, no el de humo
)
try:
    print(f"qdrant · version del servidor: {indice.server_version()}")
    ACEPTACION = run_acceptance(
        indice, puntos_completo, dim=DIM, metric="cosine",
        batch_size=LOTE, top_k=TOP_K, n_canarios=3,
    )
finally:
    indice.close()

INDICE_DIR = Path("..") / "artifacts" / "indice"
save_smoke(INDICE_DIR, motor="qdrant", results=ACEPTACION)
smoke_display(ACEPTACION, motor="qdrant")

qdrant · version del servidor: 1.18.2


motor,paso,comprobacion,que_ha_hecho,esperado,observado,resultado,segundos
qdrant,1,Colección con dimensión y métrica explícitas,"create_collection(dim=768, metric='cosine', recreate=False)","dim=768, métrica=cosine","'aurum_catalogo__gemini_embedding_2__A4__768' lista (dim=768, métrica=cosine)",✅ pasa,1.370000
qdrant,2,Ingesta por lotes,"upsert(15000 puntos, batch_size=128) → count()",count() == 15000,count() = 15000 · 317.2 vectores/s en 47.3 s,✅ pasa,47.293000
qdrant,3,Índice al día antes de aceptar consultas,index_ready() · espera con tope de 180 s,el motor declara su índice construido,AÚN INDEXANDO al terminar la ingesta · listo tras 4.1 s de espera (3 sondeos),✅ pasa,4.139000
qdrant,4,Dimensión declarada == dimensión real,collection_dim(),la colección dice 768,la colección declara 768,✅ pasa,nan
qdrant,5,Ingesta repetida sin duplicar,upsert(LOS MISMOS 15000 puntos) → count(),count() sigue en 15000,count() = 15000,✅ pasa,70.761000
qdrant,6,Canarios: cada punto se recupera a sí mismo,"search(vector del propio punto, top_k=10) × 3",los 3 vuelven en la posición 1,3/3 en la posición 1,✅ pasa,nan
qdrant,7,Persistencia tras reinicio,make motor-down MOTOR=qdrant && make motor-up MOTOR=qdrant (+ relectura),mismo count() y los mismos canarios en la posición 1,(pendiente),✍️ manual,nan
qdrant,8,Recursos con el índice completo,make motor-stats (con el motor vivo y los 15.000 ingeridos),"RAM del contenedor y tamaño del volumen, esta vez sobre volumen limpio",(pendiente),✍️ manual,nan


## G.3 · Paso 7 · persistencia, con los 15.000 dentro

La misma comprobación que el paso 8 de la prueba de humo, pero sobre el índice que se entrega. Ejecuta la celda, reinicia, y vuelve a ejecutarla con la otra etiqueta:

```bash
make motor-down MOTOR=qdrant && make motor-up MOTOR=qdrant
```

Se lee con el vector del primer canario, así que comprueba dos cosas a la vez: que el recuento sobrevive y que ese punto sigue encontrándose a sí mismo donde estaba.

In [11]:
INDICE_PERSISTENCIA = globals().get("INDICE_PERSISTENCIA") or {}


def relectura(etiqueta):
    """PASO 7 · Solo lee. No crea, no ingiere y no borra."""
    store = QdrantStore(
        collection=COLECCION,
        url=os.environ.get("AURUM_QDRANT_URL", "http://localhost:6333"),
        api_key=os.environ.get("AURUM_QDRANT_API_KEY"),
        prefix=CATALOG_PREFIX,
    )
    try:
        INDICE_PERSISTENCIA[etiqueta] = read_snapshot(
            store, canarios[0].vector, top_k=TOP_K
        )
    finally:
        store.close()

    actual = INDICE_PERSISTENCIA[etiqueta]
    print(f"{etiqueta}: count={actual.count} · top-1 = {actual.ids[0]}")
    print(f"  ¿el canario se encuentra a si mismo? "
          f"{actual.ids[0] == canarios[0].record_id}")
    antes, despues = (INDICE_PERSISTENCIA.get(e) for e in ("antes", "despues"))
    if not (antes and despues):
        print("  (falta la otra mitad: reinicia y llama con la otra etiqueta)")
        return
    check = persistence_check(antes, despues)
    print(f"\n  {check.verdict()}")
    record_manual(ACEPTACION, 7, observed=check.verdict(), passed=check.passed,
                  manual_steps=ACCEPTANCE_MANUAL_NUMBERS)
    save_smoke(INDICE_DIR, motor="qdrant", results=ACEPTACION)
    print(f"  → fila 7 anotada: {'✅ pasa' if check.passed else '❌ falla'}")


relectura("antes")

antes: count=15000 · top-1 = e1a0e559-6a49-5be5-b617-ec8a4899e975
  ¿el canario se encuentra a si mismo? True
  (falta la otra mitad: reinicia y llama con la otra etiqueta)


In [12]:
relectura("despues")

despues: count=15000 · top-1 = e1a0e559-6a49-5be5-b617-ec8a4899e975
  ¿el canario se encuentra a si mismo? True

  ✅ idéntico — count() 15000 y los mismos 10 ids en el mismo orden
  → fila 7 anotada: ✅ pasa


## G.4 · Paso 8 · recursos, esta vez los que van al README

Con el motor vivo y los 15.000 dentro:

```bash
make motor-stats
```

La salida entera a `artifacts/recursos/qdrant_indice.txt`. Con el volumen borrado antes de construir, el tamaño que salga es el del índice y de nada más.

> ⚠️ **Se toma justo después de G.2, antes del reinicio de G.3.** Un contenedor recién reiniciado y sin tráfico no ha tocado sus datos, y su cifra no es la del índice sirviendo: la primera medición salió a 28,7 MiB con **6,72 kB** de `NET I/O` —menos memoria que con 1.500 puntos, que no puede ser—. Si esa columna está a cero, la foto no vale.

Y aun así es RAM **en reposo tras ingerir**, no bajo carga: la huella sirviendo consultas sale gratis en NB06, con el bucle de latencia corriendo.

In [15]:
# 📄 DATOS · artifacts/recursos/qdrant_indice.txt
from aurum.motores import resource_row

transcrito = RECURSOS / "qdrant_indice.txt"
if not transcrito.exists():
    print(f"Falta {transcrito}: pega ahi la salida de `make motor-stats`.")
else:
    fila_recursos = resource_row(
        transcrito.read_text(encoding="utf-8"), motor="qdrant"
    ).iloc[0]
    record_manual(ACEPTACION, 8, observed=resource_note(fila_recursos),
                  passed=bool(fila_recursos["dentro_del_limite"]),
                  manual_steps=ACCEPTANCE_MANUAL_NUMBERS)
    save_smoke(INDICE_DIR, motor="qdrant", results=ACEPTACION)
    print(resource_note(fila_recursos))

RAM 106.6 MiB en 1 contenedor · volumen 289.5 MB (no comparable entre motores) · dentro del `mem_limit` declarado, el más apretado aurum-market-qdrant al 5% · foto con el contenedor «Up 4 minutes (healthy)»


## G.5 · El artefacto del índice

La tabla de aceptación con el contrato de la colección delante. Es lo que un corrector necesita para saber qué se construyó y con qué se comprobó, sin abrir el notebook.

In [17]:
destino_indice = Path("..") / "artifacts" / "indice_catalogo.md"
tabla_aceptacion = smoke_table(ACEPTACION, motor="qdrant").drop(columns="motor")
pendientes = [r.step for r in ACEPTACION if r.passed is None]

contrato = [
    "# El índice del catálogo · aceptación (NB04 § G)",
    "",
    f"Colección: `{COLECCION}`  ·  motor: **qdrant** (R03)",
    "",
    "| Elemento | Valor | De dónde sale |",
    "|---|---|---|",
    f"| Id del punto | `record_id` (UUIDv5) | README_DATOS · idempotencia por id |",
    f"| Modelo | `{MODELO}` [{CONTRATO}] | R02 |",
    f"| Plantilla | `{PLANTILLA}` | R01 |",
    f"| Dimensión | {DIM} (truncada de 3.072 y renormalizada) | D09b |",
    f"| Métrica | cosine | D10 |",
    f"| Payload | `{ESQUEMA}` · nulos `{POLITICA_NULOS}` | D13 · D14 |",
    f"| Lote de ingesta | {LOTE} | D15 |",
    f"| Índices de payload | brand_normalized (keyword) · color_normalized (texto) | Sección B |",
    f"| Puntos | {len(puntos_completo):,} | catalogo_productos.csv |".replace(",", "."),
    "",
    "## Cómo leer los números del paso 8",
    "",
    "- La **RAM** es en reposo tras ingerir, no bajo carga. Vale solo si el "
    "`NET I/O` del transcrito no está a cero: un contenedor recién "
    "reiniciado no ha tocado sus datos y da una cifra que no es la del "
    "índice. La huella sirviendo consultas se mide en NB06.",
    "- El **volumen** no lo llena el índice. Con los 1.500 de la prueba de "
    "humo eran 293,8 MB y con los 15.000 son 289,5: diez veces más puntos y "
    "no crece. Los vectores son 46 MB; el resto es asignación del motor "
    "—WAL— y por eso escribir esos 46 MB costó 665 MB de `BLOCK I/O`.",
    "",
    "## Las ocho comprobaciones",
    "",
]
if pendientes:
    contrato += [f"> ⚠️ Pasos sin medir todavía: {pendientes}.", ""]

destino_indice.write_text(
    "\n".join(contrato) + "\n" + tabla_aceptacion.to_markdown(index=False) + "\n",
    encoding="utf-8",
)
print(f"Escrito {destino_indice}")
tabla_aceptacion

Escrito ..\artifacts\indice_catalogo.md


,paso,comprobacion,que_ha_hecho,esperado,observado,resultado,segundos
0,1,Colección con dimensión y métrica explícitas,"create_collection(dim=768, metric='cosine', re...","dim=768, métrica=cosine",'aurum_catalogo__gemini_embedding_2__A4__768' ...,✅ pasa,1.370
1,2,Ingesta por lotes,"upsert(15000 puntos, batch_size=128) → count()",count() == 15000,count() = 15000 · 317.2 vectores/s en 47.3 s,✅ pasa,47.293
2,3,Índice al día antes de aceptar consultas,index_ready() · espera con tope de 180 s,el motor declara su índice construido,AÚN INDEXANDO al terminar la ingesta · listo t...,✅ pasa,4.139
3,4,Dimensión declarada == dimensión real,collection_dim(),la colección dice 768,la colección declara 768,✅ pasa,NaN
4,5,Ingesta repetida sin duplicar,upsert(LOS MISMOS 15000 puntos) → count(),count() sigue en 15000,count() = 15000,✅ pasa,70.761
5,6,Canarios: cada punto se recupera a sí mismo,"search(vector del propio punto, top_k=10) × 3",los 3 vuelven en la posición 1,3/3 en la posición 1,✅ pasa,NaN
6,7,Persistencia tras reinicio,make motor-down MOTOR=qdrant && make motor-up ...,mismo count() y los mismos canarios en la posi...,✅ idéntico — count() 15000 y los mismos 10 ids...,✅ pasa,NaN
7,8,Recursos con el índice completo,make motor-stats (con el motor vivo y los 15....,"RAM del contenedor y tamaño del volumen, esta ...",RAM 106.6 MiB en 1 contenedor · volumen 289.5 ...,✅ pasa,NaN
